# Cyber-Quack-14B — Colab chat v2: BF16 → **Q5_K_M** quantizer • max context • max reasoning • web search (Parallel MCP) • public link

**How to use**
1. **Runtime → Change runtime type → T4 GPU** (preselected if you opened this file in Colab).
2. **One-time Hugging Face step:** `jabbatheduck/Cyber-Quack-14B` is a **gated** repo. Open the model page, sign in, click **Agree and access repository** (approval is automatic), create a READ token at `huggingface.co/settings/tokens`, and add it as the Colab secret **HF_TOKEN** (🔑 icon on the left, switch *Notebook access* ON). Cell 1 now *really* tests this before the long build, so a missing token fails in seconds.
3. **Cell 1** — pick the quant (default **Q5_K_M**) and run it. It mounts Google Drive (optional build cache), builds llama.cpp, downloads the **BF16 safetensors (~29.5 GB)**, converts them to an F16 GGUF and **quantizes to Q5_K_M (~10.5 GB)**, deleting every intermediate as it goes (**≈35–55 min** the first time; re-runs skip every finished stage).
4. **Cell 2** — keep the defaults (**Max** context, **Max** reasoning, **Parallel** web search) or change them, then run it. It shows the chat right in the notebook, plus a private link and a **public link**.
5. **Keep this tab open and leave Cell 2 running.** It watches the model server and the link and repairs them automatically.

**Good to know**
- **Why Q5_K_M:** Q8_0 (15.7 GB) can never fit a 15 GB T4, and Q6_K (12.1 GB) leaves almost no room for context. Q5_K_M (≈9.8 GiB) leaves ~5 GiB for the chat memory: **32,768 tokens with an 8-bit KV cache, or 65,536 with a 4-bit one**.
- **Context:** *Max* = the largest window that physically fits. For this model that is **65,536** (4-bit chat memory, YaRN ×2 beyond the native 32,768). For the best quality choose **32768** in Cell 2 (8-bit chat memory, no YaRN). Anything that does not fit is stepped down automatically, and you are told.
- **Reasoning:** Cyber-Quack (Qwen2.5 base) has no native thinking mode, so the notebook adds one: a reasoning protocol in the system prompt (the model thinks inside `<think>…</think>`, shown in a collapsible panel) plus, for High/Medium/Low, a hard thinking budget enforced by llama-server. **Max = unlimited thinking.** Sampling defaults are calm (temp 0.7) because high temperatures wreck tool-call JSON.
- **Web search:** the model gets `web_search` and `web_fetch` (Parallel + Exa + DuckDuckGo with auto-fallback; the Parallel key also adds the slow `web_research` tool — the keys are pre-filled). Results are capped relative to the context size and every call is time-bounded. The first search asks permission — click **Always allow** — or use the **Allow tool calls: ON / OFF** switch inside the chat box, right after the + button.
- **Long input:** 1,000+ lines of text or code is fine — paste it or attach the file. When a chat outgrows the window, the oldest turns are recycled but the system prompt is always kept.
- **Public link:** a **Cloudflare** link (no account, new address each run). The chat inside the notebook always works too.
- **No vision:** no image projector exists for this model, so this notebook is text-only.
- Chats are stored in your browser tab. If the public link changes, it's a new origin, so export chats you care about from the UI settings.
- **Optional:** tick *SAVE_MODEL_TO_DRIVE* in Cell 1 to keep the finished ~10 GB GGUF on your Drive (needs ~11 GB free) and skip the whole download+quantize next session.

In [ ]:
#@title 1️⃣ Setup — build llama.cpp, download Cyber-Quack-14B (BF16) and quantize it to Q5_K_M (≈35–55 min first run) { display-mode: "form" }
MODEL_QUANT = "Q5_K_M · ~10.5 GB · best fit for one T4 (default)" #@param ["Q5_K_M · ~10.5 GB · best fit for one T4 (default)", "Q4_K_M · ~9.0 GB · most room for context", "Q6_K · ~12.1 GB · near-lossless, small context only"]
USE_DRIVE_CACHE = True #@param {type:"boolean"}
SAVE_MODEL_TO_DRIVE = False #@param {type:"boolean"}
# MODEL_QUANT: there is NO pre-built GGUF of this model on the Hub, so this cell downloads the BF16 safetensors
#   (~29.5 GB), converts them to GGUF and quantizes to the chosen type right here (Q5_K_M by default).
#   Q5_K_M (≈9.8 GiB) is the sweet spot on a free 15 GB T4: it leaves room for a 32k-64k context.
# USE_DRIVE_CACHE: mounts Google Drive and caches the compiled llama.cpp binaries there, so the next
#   session skips the ~15 min build.
# SAVE_MODEL_TO_DRIVE: also keeps the finished ~10 GB GGUF on Drive so the next session skips the whole
#   35-55 min download+convert+quantize (needs ~11 GB free on your Drive; off by default).
# ⚠️ GATED REPO: jabbatheduck/Cyber-Quack-14B requires a Hugging Face token that has accepted its terms.
#   Accept once on the model page, then add a READ token as the Colab secret HF_TOKEN (🔑 icon on the left)
#   and switch "Notebook access" ON. Details in the markdown above.

import os, sys, time, json, shutil, subprocess
from concurrent.futures import ThreadPoolExecutor

SCRATCH = "/content"
QUANT      = MODEL_QUANT.split()[0]                     # "Q5_K_M" | "Q4_K_M" | "Q6_K"
REPO_ID    = "jabbatheduck/Cyber-Quack-14B"             # BF16 safetensors only (no pre-built GGUF)
LLAMA_TAG  = "b11925"                                   # pinned llama.cpp release (exists; verified), so upstream changes can't break this notebook
WITH_QUANTIZE = True

LLAMA_DIR  = "/content/llama.cpp"
BIN_PATH   = f"{LLAMA_DIR}/build/bin/llama-server"
QBIN_PATH  = f"{LLAMA_DIR}/build/bin/llama-quantize"
STAMP      = f"{LLAMA_DIR}/build/cyberquack_build_stamp.txt"
MODEL_DIR  = f"{SCRATCH}/models"
BF16_DIR   = f"{MODEL_DIR}/cyber-quack-bf16"
F16_GGUF   = f"{SCRATCH}/cyber-quack-f16.gguf"
Q8_GGUF    = f"{SCRATCH}/cyber-quack-q8_0.gguf"          # only used by the low-disk rescue path
FINAL_GGUF = f"{SCRATCH}/cyber-quack-{QUANT.lower()}.gguf"
CF_PATH    = "/content/cloudflared"
CONFIG     = "/content/cyberquack_config.json"
BUILD_LOG  = "/content/build.log"
CF_LOG     = "/content/cf.log"
CONVERT_LOG = "/content/convert.log"
QUANT_LOG  = "/content/quantize.log"
UI_TGZ     = "/content/llama-ui.tar.gz"
WEBUI_SRC  = "/content/webui_src"
CACHE_DIR  = "/content/drive/MyDrive/cyber_quack_cache"
EXPECT_GB  = {"Q5_K_M": 10.5, "Q4_K_M": 9.0, "Q6_K": 12.1}[QUANT]
os.environ["PATH"] += ":/usr/local/cuda/bin"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.makedirs(MODEL_DIR, exist_ok=True)

# ==== PURE-BEGIN  (no side effects: unit-tested outside Colab) ====
import os, re, json, struct

# ---------------------------------------------------------------- chat templates
# One template body, two tool-response styles. Both render the tools block + <tool_call> format that
# llama.cpp's Hermes-2-Pro parser understands, and both know about `enable_thinking`, so the server can
# split <think>...</think> into the reasoning panel (and --reasoning off closes the think block at once).
_TEMPLATE_HEAD = r"""{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages and messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- '__DEFAULT_SYSTEM__' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages and messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
    {%- else %}
        {{- '<|im_start|>system\n__DEFAULT_SYSTEM__<|im_end|>\n' }}
    {%- endif %}
{%- endif %}
{%- for message in messages %}
    {%- if (message.role == "user") or (message.role == "system" and not loop.first) or (message.role == "assistant" and not message.tool_calls) %}
        {{- '<|im_start|>' + message.role + '\n' + message.content + '<|im_end|>' + '\n' }}
    {%- elif message.role == "assistant" %}
        {{- '<|im_start|>' + message.role }}
        {%- if message.content %}
            {{- '\n' + message.content }}
        {%- endif %}
        {%- for tool_call in message.tool_calls %}
            {%- if tool_call.function is defined %}
                {%- set tool_call = tool_call.function %}
            {%- endif %}
            {{- '\n<tool_call>\n{"name": "' }}
            {{- tool_call.name }}
            {{- '", "arguments": ' }}
            {{- tool_call.arguments | tojson }}
            {{- '}\n</tool_call>' }}
        {%- endfor %}
        {{- '<|im_end|>\n' }}
"""
_TOOL_QWEN = r"""    {%- elif message.role == "tool" %}
        {%- if (loop.index0 == 0) or (messages[loop.index0 - 1].role != "tool") %}
            {{- '<|im_start|>user' }}
        {%- endif %}
        {{- '\n<tool_response>\n' }}
        {{- message.content }}
        {{- '\n</tool_response>' }}
        {%- if loop.last or (messages[loop.index0 + 1].role != "tool") %}
            {{- '<|im_end|>\n' }}
        {%- endif %}
    {%- endif %}
"""
_TOOL_HERMES = r"""    {%- elif message.role == "tool" %}
        {%- if (loop.index0 == 0) or (messages[loop.index0 - 1].role != "tool") %}
            {{- '<|im_start|>tool' }}
        {%- endif %}
        {{- '\n<tool_response>\n' }}
        {{- message.content }}
        {{- '\n</tool_response>' }}
        {%- if loop.last or (messages[loop.index0 + 1].role != "tool") %}
            {{- '<|im_end|>\n' }}
        {%- endif %}
    {%- endif %}
"""
_TEMPLATE_TAIL = r"""{%- endfor %}
{%- if add_generation_prompt %}
    {{- '<|im_start|>assistant\n' }}
    {%- if enable_thinking is defined and enable_thinking is false %}
        {{- '<think>\n\n</think>\n\n' }}
    {%- endif %}
{%- endif %}
"""

def build_chat_template(style, default_system):
    body = _TEMPLATE_HEAD + (_TOOL_HERMES if style == "hermes" else _TOOL_QWEN) + _TEMPLATE_TAIL
    return body.replace("__DEFAULT_SYSTEM__", default_system.replace("'", ""))


# ---------------------------------------------------------------- minimal GGUF header reader
_SCALARS = {0: "<B", 1: "<b", 2: "<H", 3: "<h", 4: "<I", 5: "<i", 6: "<f", 7: "<?", 10: "<Q", 11: "<q", 12: "<d"}

def read_gguf_meta(path, want_prefixes=("general.", "tokenizer.chat_template"), buf_mb=96):
    """Return {key: value} for the small metadata keys (scalars + strings); big arrays are skipped.
    Pure python, no dependency. Raises ValueError if the file is not a GGUF."""
    with open(path, "rb") as f:
        buf = f.read(buf_mb << 20)
    if buf[:4] != b"GGUF":
        raise ValueError("not a GGUF file")
    ver = struct.unpack_from("<I", buf, 4)[0]
    if ver < 2:
        raise ValueError(f"unsupported GGUF version {ver}")
    n_kv = struct.unpack_from("<Q", buf, 16)[0]
    pos, out = 24, {}

    def rstr(p):
        n = struct.unpack_from("<Q", buf, p)[0]
        return buf[p + 8:p + 8 + n].decode("utf-8", "replace"), p + 8 + n

    def skip(t, p):
        if t in _SCALARS:
            return p + struct.calcsize(_SCALARS[t])
        if t == 8:
            n = struct.unpack_from("<Q", buf, p)[0]
            return p + 8 + n
        if t == 9:
            et, cnt = struct.unpack_from("<IQ", buf, p)
            p += 12
            if et in _SCALARS:
                return p + cnt * struct.calcsize(_SCALARS[et])
            for _ in range(cnt):
                p = skip(et, p)
            return p
        raise ValueError(f"bad GGUF value type {t}")

    for _ in range(n_kv):
        key, pos = rstr(pos)
        t = struct.unpack_from("<I", buf, pos)[0]
        pos += 4
        keep = key.startswith(want_prefixes) or key.endswith((".block_count", ".context_length",
               ".head_count_kv", ".head_count", ".embedding_length", ".key_length"))
        if keep and t in _SCALARS:
            out[key] = struct.unpack_from(_SCALARS[t], buf, pos)[0]
            pos += struct.calcsize(_SCALARS[t])
        elif keep and t == 8:
            out[key], pos = rstr(pos)
        else:
            pos = skip(t, pos)
    return out

def model_dims(meta, fallback):
    """(n_layer, n_kv_head, head_dim, n_ctx_train) from GGUF metadata, falling back to the profile."""
    arch = meta.get("general.architecture", "")
    g = lambda k: meta.get(f"{arch}.{k}")
    try:
        n_layer = int(g("block_count") or fallback["N_LAYER"])
        kvh = g("attention.head_count_kv")
        kvh = int(kvh) if isinstance(kvh, (int, float)) and kvh else fallback["N_KV_HEAD"]
        hd = g("attention.key_length")
        if not hd:
            emb, heads = g("embedding_length"), g("attention.head_count")
            hd = (int(emb) // int(heads)) if emb and heads else fallback["HEAD_DIM"]
        return n_layer, kvh, int(hd), int(g("context_length") or 0)
    except Exception:
        return fallback["N_LAYER"], fallback["N_KV_HEAD"], fallback["HEAD_DIM"], 0


# ---------------------------------------------------------------- GPU memory planner
KV_BYTES_PER_ELEM = {"f16": 2.0, "q8_0": 34 / 32, "q4_0": 18 / 32}   # ggml block sizes

def kv_mib(ctx, kv, n_layer, n_kv_head, head_dim):
    return 2 * n_layer * n_kv_head * head_dim * KV_BYTES_PER_ELEM[kv] * ctx / 1048576

def plan_candidates(model_mib, total_mib, ladder, target, kv_pref, dims, overhead_mib, min_free_mib):
    """Ordered (ctx, kv) launch attempts. Largest context first; at equal context the best KV precision first.
    Entries predicted to fit come first; the rest (a safety net if the estimate is too pessimistic) follow."""
    kv_order = {"auto": ["q8_0", "q4_0"], "q8_0": ["q8_0"], "q4_0": ["q4_0"]}[kv_pref]
    sizes = [c for c in ladder if c <= target]
    fit, nofit = [], []
    for c in sizes:
        for kv in kv_order:
            need = model_mib + kv_mib(c, kv, *dims) + overhead_mib + min_free_mib
            (fit if need <= total_mib else nofit).append((c, kv))
    return fit + [x for x in nofit if x[1] == kv_order[-1]][:3]

def yarn_args(ctx, yarn_base):
    """llama-server flags for contexts beyond the model's trained window (Qwen2.5 + YaRN)."""
    if not yarn_base or ctx <= yarn_base:
        return []
    scale = round(ctx / yarn_base * 2) / 2          # 1.5, 2.0, ...
    return ["--rope-scaling", "yarn", "--rope-scale", str(scale), "--yarn-orig-ctx", str(yarn_base)]

def tool_cap_chars(ctx):
    """Max characters one web-tool result may add to the conversation (~1/8 of the window, 3 chars/token)."""
    return int(min(30000, max(5000, ctx * 0.375)))


# ---------------------------------------------------------------- web UI tarball (any archive layout)
def extract_ui(tgz, dest):
    """Unpack llama-<tag>-ui.tar.gz into `dest`, whatever the top-level layout is: the directory that
    holds index.html (shallowest one) becomes `dest`."""
    import tarfile, shutil
    tmp = dest + ".tmp"
    shutil.rmtree(tmp, ignore_errors=True)
    os.makedirs(tmp)
    with tarfile.open(tgz) as t:
        try:
            t.extractall(tmp, filter="data")
        except TypeError:                       # Python without extraction filters
            t.extractall(tmp)
    root, best = None, 10**6
    for r, _dirs, files in os.walk(tmp):
        if "index.html" in files:
            depth = os.path.relpath(r, tmp).count(os.sep)
            if depth < best:
                best, root = depth, r
    if root is None:
        shutil.rmtree(tmp, ignore_errors=True)
        raise RuntimeError("the UI archive has no index.html")
    shutil.rmtree(dest, ignore_errors=True)
    shutil.copytree(root, dest)
    shutil.rmtree(tmp, ignore_errors=True)
    return dest


# ---------------------------------------------------------------- choose a GGUF file in a repo listing
def pick_gguf(files, quant, preference, exact_name=None):
    """-> (filename, quant_used) or None. Exact name first, then the requested quant, then the next best."""
    ggufs = sorted(f for f in files if f.lower().endswith(".gguf") and "mmproj" not in f.lower()
                   and "/" not in f and not re.search(r"-\d{5}-of-\d{5}", f))
    if exact_name and exact_name in ggufs:
        return exact_name, quant
    order = preference[preference.index(quant):] if quant in preference else list(preference)
    for q in order:
        pat = re.compile(r"(?<![A-Za-z0-9])" + re.escape(q) + r"(?![A-Za-z0-9])", re.I)
        hit = [f for f in ggufs if pat.search(f)]
        if hit:
            return hit[0], q
    return None
# ==== PURE-END ====


# ======================= shared helpers (identical in both notebooks) =======================
import threading
SRC_LOCK = threading.Lock()
BUILD_STATE = {"done": threading.Event(), "ok": False, "err": ""}

def run(cmd, log, env=None, retries=1):
    for attempt in range(retries):
        with open(log, "a") as f:
            f.write(f"\n$ {' '.join(cmd)}\n"); f.flush()
            p = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, env=env)
        if p.returncode == 0:
            return
        if attempt + 1 < retries:
            time.sleep(5)
    tail = open(log, errors="replace").read()[-4000:]
    raise RuntimeError(f"Command failed: {' '.join(cmd)}\n\n{tail}")

def binary_ok(path):
    try:
        out = subprocess.run([path, "--version"], capture_output=True, text=True, timeout=60)
        return out.returncode == 0
    except Exception:
        return False

def find_libcuda():
    """Path of the CUDA driver library (libcuda.so / libcuda.so.1), or '' if absent."""
    paths = []
    try:
        out = subprocess.run(["ldconfig", "-p"], capture_output=True, text=True).stdout or ""
        for line in out.splitlines():
            if "libcuda.so" in line and "=>" in line:
                paths.append(line.rsplit("=> ", 1)[-1].strip())
    except Exception:
        pass
    paths += ["/usr/lib/x86_64-linux-gnu/libcuda.so.1", "/usr/lib64/libcuda.so.1",
              "/usr/local/cuda/compat/libcuda.so.1",
              "/usr/local/cuda/lib64/stubs/libcuda.so",
              "/usr/local/cuda/targets/x86_64-linux/lib/stubs/libcuda.so"]
    for p in paths:
        if p and os.path.exists(p):
            return p
    return ""

def ensure_cuda_driver_link():
    """CMake's FindCUDAToolkit sometimes cannot see the driver library (CUDA::cuda_driver); a libcuda.so
    symlink next to the toolkit libs fixes that. No-op when CMake can already find it."""
    need = [d for d in ("/usr/local/cuda/lib64", "/usr/local/cuda/targets/x86_64-linux/lib")
            if os.path.isdir(d) and not os.path.exists(f"{d}/libcuda.so")]
    if not need:
        return ""
    drv = find_libcuda()
    if not drv:
        return ""
    for d in need:
        try:
            os.symlink(drv, f"{d}/libcuda.so")
            print(f"🔧 linked {drv} → {d}/libcuda.so (CUDA::cuda_driver fix)")
        except OSError:
            pass
    return drv

def ensure_llama_source():
    """Pinned llama.cpp checkout (needed by the compiler and, for Cyber-Quack, by convert_hf_to_gguf.py).
    Thread-safe and idempotent: never deletes a checkout that is already in use."""
    with SRC_LOCK:
        marker = f"{LLAMA_DIR}/.pinned_tag"
        ok = (os.path.exists(f"{LLAMA_DIR}/CMakeLists.txt") and os.path.exists(marker)
              and open(marker).read().strip() == LLAMA_TAG)
        if ok:
            return "already there"
        shutil.rmtree(LLAMA_DIR, ignore_errors=True)
        open(BUILD_LOG, "a").close()
        run(["git", "clone", "--depth", "1", "--branch", LLAMA_TAG,
             "https://github.com/ggml-org/llama.cpp", LLAMA_DIR], BUILD_LOG, retries=3)
        open(marker, "w").write(LLAMA_TAG)
        return "cloned"

def prepare_webui():
    """Download + unpack the official prebuilt chat UI of this exact release (no Node.js needed).
    Cell 2 patches a copy of it and serves it with --path, so the server build never depends on it."""
    if os.path.isfile(f"{WEBUI_SRC}/index.html"):
        return "already there"
    url = f"https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_TAG}/llama-{LLAMA_TAG}-ui.tar.gz"
    run(["wget", "-q", "--tries=3", "-O", UI_TGZ, url], BUILD_LOG)
    extract_ui(UI_TGZ, WEBUI_SRC)
    return "downloaded"

def build_server():
    want = f"{LLAMA_TAG} sm{ARCH}"
    def good():
        s = os.path.exists(BIN_PATH) and binary_ok(BIN_PATH)
        q = (not WITH_QUANTIZE) or (os.path.exists(QBIN_PATH) and binary_ok(QBIN_PATH))
        return s and q
    if good() and os.path.exists(STAMP) and open(STAMP).read().strip() == want:
        return "already built"
    if USE_CACHE:
        os.makedirs(os.path.dirname(BIN_PATH), exist_ok=True)
        if os.path.exists(CACHED_BIN):
            shutil.copy(CACHED_BIN, BIN_PATH); os.chmod(BIN_PATH, 0o755)
        if WITH_QUANTIZE and os.path.exists(CACHED_QUANT):
            shutil.copy(CACHED_QUANT, QBIN_PATH); os.chmod(QBIN_PATH, 0o755)
        if good():
            open(STAMP, "w").write(want)
            return "restored from Drive cache"
    ensure_llama_source()
    env = dict(os.environ)
    drv = ensure_cuda_driver_link()
    # -DLLAMA_BUILD_UI=OFF: the chat UI is served from files (--path), so a UI download/embedding problem
    # can never break the compile.
    cmake_cmd = ["cmake", "-S", LLAMA_DIR, "-B", f"{LLAMA_DIR}/build",
                 "-DGGML_CUDA=ON", f"-DCMAKE_CUDA_ARCHITECTURES={ARCH}",
                 "-DBUILD_SHARED_LIBS=OFF", "-DLLAMA_OPENSSL=OFF", "-DLLAMA_BUILD_UI=OFF",
                 "-DLLAMA_BUILD_TESTS=OFF", "-DLLAMA_BUILD_EXAMPLES=OFF",
                 "-DCMAKE_BUILD_TYPE=Release"]
    try:
        run(cmake_cmd, BUILD_LOG, env=env)
    except RuntimeError as e:
        if not (drv and "cuda_driver" in str(e)):
            raise
        run(cmake_cmd + [f"-DCUDA_DRIVER_LIBRARY={drv}"], BUILD_LOG, env=env)
    targets = ["--target", "llama-server"] + (["--target", "llama-quantize"] if WITH_QUANTIZE else [])
    run(["cmake", "--build", f"{LLAMA_DIR}/build", "--config", "Release"] + targets +
        ["-j", str(min(os.cpu_count() or 2, 4))], BUILD_LOG, env=env)
    if not binary_ok(BIN_PATH):
        raise RuntimeError(f"The server was built but does not start. See {BUILD_LOG}")
    if WITH_QUANTIZE and not binary_ok(QBIN_PATH):
        raise RuntimeError(f"llama-quantize was not built. See {BUILD_LOG}")
    open(STAMP, "w").write(want)
    if USE_CACHE:
        try:
            shutil.copy(BIN_PATH, CACHED_BIN)
            if WITH_QUANTIZE:
                shutil.copy(QBIN_PATH, CACHED_QUANT)
        except OSError as e:
            print(f"⚠️ could not save the build to Drive ({e}) - continuing")
    return "built from source"

def build_job():
    """build_server() + signal for the threads that need the binaries (quantize stage)."""
    try:
        res = build_server()
        BUILD_STATE["ok"] = True
        return res
    except BaseException as e:      # noqa: BLE001
        BUILD_STATE["err"] = str(e)[:500]
        raise
    finally:
        BUILD_STATE["done"].set()

def get_cloudflared():
    if os.path.exists(CF_PATH) and os.path.getsize(CF_PATH) > 1_000_000 and binary_ok(CF_PATH):
        return "already there"
    run(["wget", "-q", "--tries=3", "-O", CF_PATH,
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
        CF_LOG, retries=3)
    os.chmod(CF_PATH, 0o755)
    if os.path.getsize(CF_PATH) < 1_000_000 or not binary_ok(CF_PATH):
        os.remove(CF_PATH)
        raise RuntimeError("cloudflared download looks broken; run this cell again.")
    return "downloaded"

PIP_LOCK = threading.Lock()      # two pip processes at once can corrupt each other

def pip_install(pkgs):
    with PIP_LOCK:
        p = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-input"] + pkgs,
                           capture_output=True, text=True)
    return p.returncode == 0, (p.stderr or p.stdout or "").strip()[-200:]

def get_web_deps():
    # DuckDuckGo search + page-to-text: used by the DuckDuckGo option and as automatic fallback
    ok, err = pip_install(["ddgs", "trafilatura"])
    return "installed" if ok else "skipped (pip failed; Parallel/Exa still work)"

def hf_token():
    try:
        from google.colab import userdata
        return (userdata.get("HF_TOKEN") or "").strip()
    except Exception:
        return ""

class Heartbeat(threading.Thread):
    """Prints a progress line every few seconds while a download/conversion runs."""
    def __init__(self, label, watch, is_dir, interval=10, total_bytes=0):
        super().__init__(daemon=True)
        self.label, self.watch, self.is_dir = label, watch, is_dir
        self.interval, self.total = interval, total_bytes
        self.t0 = time.time()
        self.stop_event = threading.Event()

    def _size(self):
        try:
            if self.is_dir:
                total = 0
                for root, _, files in os.walk(self.watch):
                    for name in files:
                        try:
                            total += os.path.getsize(os.path.join(root, name))
                        except OSError:
                            pass
                return total
            return os.path.getsize(self.watch) if os.path.exists(self.watch) else 0
        except OSError:
            return 0

    def run(self):
        while not self.stop_event.wait(self.interval):
            cur = self._size()
            el = int(time.time() - self.t0)
            msg = f"   [{self.label}] {el // 60}:{el % 60:02d} | {cur/1e9:5.1f} GB"
            if self.total:
                msg += f" / ~{self.total/1e9:.0f} GB"
            try:
                msg += f" | {shutil.disk_usage(SCRATCH).free/1e9:5.1f} GB free"
            except OSError:
                pass
            print(msg, flush=True)

    def stop(self):
        self.stop_event.set()
        self.join(timeout=5)

def run_with_progress(cmd, log, label, watch_path, is_dir, interval):
    """Run a subprocess while printing progress (file/dir size + free disk) every `interval` s."""
    with open(log, "a") as f:
        f.write(f"\n$ {' '.join(cmd)}\n"); f.flush()
        proc = subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT)
    hb = Heartbeat(label, watch_path, is_dir, interval)
    hb.start()
    try:
        while proc.poll() is None:
            time.sleep(1)
    finally:
        hb.stop()
    return proc.returncode

def gpu_check():
    try:
        gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name,compute_cap,memory.total",
                                       "--format=csv,noheader"], text=True).strip().splitlines()[0]
        name, cap, vram = [x.strip() for x in gpu.split(",")]
        print(f"✅ GPU: {name} (sm_{cap.replace('.', '')}, {vram})")
        return cap.replace(".", "")
    except Exception:
        raise RuntimeError("❌ No GPU found. Runtime → Change runtime type → T4 GPU → Save, then run this cell again.")

def mount_drive():
    global USE_DRIVE_CACHE
    if USE_DRIVE_CACHE:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            print("✅ Google Drive mounted (build cache)")
        except Exception as e:
            USE_DRIVE_CACHE = False
            print(f"⚠️ Could not mount Google Drive ({e}); continuing without the build cache.")
    return USE_DRIVE_CACHE

def wait_jobs(jobs, t0):
    last = -1
    while not all(j.done() for j in jobs.values()):
        time.sleep(5)
        mins = int((time.time() - t0) // 60)
        if mins != last:
            last = mins
            print(f"   {mins:>2} min | " + " | ".join(f"{k}: {'done' if j.done() else '…'}" for k, j in jobs.items()))
    errors = [f"[{k}] {j.exception()}" for k, j in jobs.items()
              if j.exception() is not None and k != "web tools"]
    if errors:
        raise RuntimeError("Setup failed — just run this cell again (finished parts are reused).\n\n" + "\n\n".join(errors))


# ======================= Cyber-Quack specific =======================
GATED_HELP = (
    "❌ jabbatheduck/Cyber-Quack-14B is a GATED Hugging Face repo: downloads need a token that has accepted its terms.\n"
    "   1. Open https://huggingface.co/jabbatheduck/Cyber-Quack-14B and sign in.\n"
    "   2. Click “Agree and access repository” (approval is automatic).\n"
    "   3. Create a READ token at https://huggingface.co/settings/tokens.\n"
    "   4. Add it to this notebook as a secret named HF_TOKEN (Colab: the 🔑 icon on the left)\n"
    "      and switch “Notebook access” ON for it.\n"
    "   5. Run this cell again.")

def _is_gated_error(e):
    m = (str(e) + " " + type(e).__name__).lower()
    return any(s in m for s in ("401", "403", "gated", "restricted", "log in", "repositorynotfound", "unauthorized"))

def check_gated_access():
    """Really verify access BEFORE the long build: HF shows gated-repo *metadata* to everyone, so
    model_info() proves nothing - fetching a file does."""
    try:
        from huggingface_hub import hf_hub_download
    except ImportError:
        pip_install(["huggingface_hub"])
        from huggingface_hub import hf_hub_download
    token = hf_token() or None
    try:
        hf_hub_download(REPO_ID, "config.json", token=token, local_dir=f"{MODEL_DIR}/_probe")
        shutil.rmtree(f"{MODEL_DIR}/_probe", ignore_errors=True)
    except Exception as e:                      # noqa: BLE001
        if _is_gated_error(e):
            raise RuntimeError(GATED_HELP + ("\n   (No HF_TOKEN secret was found in this notebook.)" if not token else "")) from e
        print(f"⚠️ Could not pre-check {REPO_ID} ({str(e)[:120]}); continuing - the download stage retries on its own.")

def download_bf16():
    """~29.5 GB of BF16 safetensors into BF16_DIR (resumable; completed files are skipped on re-run)."""
    from huggingface_hub import snapshot_download, HfApi
    token = hf_token() or None
    total = 0
    try:
        info = HfApi(token=token).model_info(REPO_ID, files_metadata=True)
        total = sum(s.size or 0 for s in info.siblings if (s.rfilename or "").endswith(".safetensors"))
    except Exception:
        total = int(29.5e9)
    allow = ["*.safetensors", "*.json", "*.txt", "*.model", "*.tiktoken", "*.jinja"]
    last = None
    for attempt in range(4):
        hb = Heartbeat("download", BF16_DIR, True, 10, total)
        hb.start()
        try:
            snapshot_download(REPO_ID, local_dir=BF16_DIR, allow_patterns=allow, token=token)
            return
        except Exception as e:                  # noqa: BLE001
            last = e
            if _is_gated_error(e):
                raise RuntimeError(GATED_HELP) from e
            print(f"⚠️ download attempt {attempt + 1}/4 failed: {str(e)[:200]}")
            time.sleep(10 * (attempt + 1))
        finally:
            hb.stop()
    raise RuntimeError(f"BF16 download failed after 4 tries: {last}")

def get_convert_deps():
    """Whatever convert_hf_to_gguf.py imports that Colab does not already have (one pip process)."""
    need = []
    for mod, pkg in (("torch", "torch"), ("transformers", "transformers"), ("safetensors", "safetensors"),
                     ("sentencepiece", "sentencepiece"), ("numpy", "numpy"), ("yaml", "pyyaml"),
                     ("google.protobuf", "protobuf")):
        try:
            __import__(mod)
        except ImportError:
            need.append(pkg)
    if not need:
        return "already available"
    cmd = list(need)
    if "torch" in need:
        cmd += ["--extra-index-url", "https://download.pytorch.org/whl/cpu"]   # CPU wheel: conversion is CPU-bound
    ok, err = pip_install(cmd)
    if not ok:
        raise RuntimeError(f"pip failed for {need}: {err}")
    return "installed " + ", ".join(need)

def valid_gguf(path):
    try:
        with open(path, "rb") as f:
            return f.read(4) == b"GGUF" and os.path.getsize(path) > 1_000_000_000
    except OSError:
        return False

def wait_for_build():
    """The quantizer comes from the compile job: wait for it instead of racing it."""
    print("   (waiting for the llama.cpp build to finish before quantizing...)")
    BUILD_STATE["done"].wait(timeout=3 * 3600)
    if not BUILD_STATE["ok"]:
        raise RuntimeError("llama.cpp did not build, so the model cannot be quantized: " + BUILD_STATE["err"])
    if not (os.path.exists(QBIN_PATH) and binary_ok(QBIN_PATH)):
        raise RuntimeError(f"llama-quantize is missing after the build. See {BUILD_LOG}")

def convert_stage():
    """BF16 safetensors -> F16 GGUF (atomic: written as .part, renamed when complete).
    Returns 'f16', or 'q8' when the disk was too small and the Q8_0 rescue path was used."""
    ensure_llama_source()
    script = f"{LLAMA_DIR}/convert_hf_to_gguf.py"
    if not os.path.exists(script):
        raise RuntimeError(f"{script} is missing (llama.cpp checkout incomplete)")
    print("⏳ Converting BF16 → F16 GGUF (CPU-bound, ~10–25 min)...")
    free0 = shutil.disk_usage(SCRATCH).free / 1e9
    print(f"   free disk before: {free0:.1f} GB")
    open(CONVERT_LOG, "w").close()
    part = F16_GGUF + ".part"
    if os.path.exists(part):
        os.remove(part)
    cmd = [sys.executable, script, BF16_DIR, "--outfile", part, "--outtype", "f16"]
    rc = run_with_progress(cmd, CONVERT_LOG, "convert", part, False, 10)
    print(f"   free disk after: {shutil.disk_usage(SCRATCH).free/1e9:.1f} GB (before: {free0:.1f} GB)")
    if rc == 0 and os.path.exists(part):
        os.replace(part, F16_GGUF)
        return "f16"
    tail = open(CONVERT_LOG, errors="replace").read()[-2500:]
    low = tail.lower()
    if "no space left" in low or "errno 28" in low or "os error 28" in low:
        if os.path.exists(part):
            os.remove(part)
        print("⚠️ Disk ran out during the F16 conversion — converting straight to Q8_0 instead "
              "(needs ~14 GB less scratch), then re-quantizing that to " + QUANT + ".")
        qpart = Q8_GGUF + ".part"
        cmd = [sys.executable, script, BF16_DIR, "--outfile", qpart, "--outtype", "q8_0"]
        if run_with_progress(cmd, CONVERT_LOG, "convert-q8", qpart, False, 10) == 0 and os.path.exists(qpart):
            os.replace(qpart, Q8_GGUF)
            return "q8"
        tail = open(CONVERT_LOG, errors="replace").read()[-2500:]
    raise RuntimeError(f"BF16 → GGUF conversion failed:\n{tail}")

def quantize_stage(src, requantize=False):
    wait_for_build()
    print(f"⏳ Quantizing {os.path.basename(src)} → {QUANT} (CPU-bound, ~8–15 min)...")
    open(QUANT_LOG, "w").close()
    part = FINAL_GGUF + ".part"
    if os.path.exists(part):
        os.remove(part)
    cmd = [QBIN_PATH] + (["--allow-requantize"] if requantize else []) + [src, part, QUANT, str(os.cpu_count() or 2)]
    rc = run_with_progress(cmd, QUANT_LOG, "quantize", part, False, 5)
    if rc != 0 or not valid_gguf(part):
        raise RuntimeError(f"Quantization failed (exit {rc}):\n" + open(QUANT_LOG, errors="replace").read()[-2500:])
    os.replace(part, FINAL_GGUF)
    print(f"✅ Quantized: {os.path.basename(FINAL_GGUF)} ({os.path.getsize(FINAL_GGUF)/1e9:.1f} GB)")

def cleanup(keep):
    """Remove intermediates and any other cyber-quack GGUFs (no orphaned multi-GB files)."""
    for name in os.listdir(SCRATCH):
        if name.startswith("cyber-quack-") and (name.endswith(".gguf") or name.endswith(".part")) and name not in keep:
            try:
                os.remove(os.path.join(SCRATCH, name)); print(f"   removed {name}")
            except OSError:
                pass
    shutil.rmtree(BF16_DIR, ignore_errors=True)

def drive_model_path():
    return f"{CACHE_DIR}/{os.path.basename(FINAL_GGUF)}"

def save_model_to_drive():
    if not (SAVE_MODEL_TO_DRIVE and USE_CACHE):
        return
    dst = drive_model_path()
    try:
        need = os.path.getsize(FINAL_GGUF)
        if os.path.exists(dst) and valid_gguf(dst):
            return
        if shutil.disk_usage("/content/drive/MyDrive").free < need + 500_000_000:
            print("⚠️ Not enough free space on Google Drive for the model copy — skipping it.")
            return
        print(f"⏳ Saving the model to Drive ({need/1e9:.1f} GB)...")
        shutil.copyfile(FINAL_GGUF, dst + ".part")
        os.replace(dst + ".part", dst)
        print("✅ Model saved to Drive — the next session restores it in a few minutes.")
    except Exception as e:                      # noqa: BLE001
        print(f"⚠️ Could not save the model to Drive ({e}) — continuing.")

def prep_model():
    """BF16 → F16 → QUANT → cleanup (each stage is skipped when its output already exists)."""
    t0 = time.time()
    if valid_gguf(FINAL_GGUF):
        print(f"✅ {os.path.basename(FINAL_GGUF)} already exists ({os.path.getsize(FINAL_GGUF)/1e9:.1f} GB) — "
              "skipping download/conversion/quantization.")
        cleanup({os.path.basename(FINAL_GGUF)})
        return FINAL_GGUF, QUANT
    if SAVE_MODEL_TO_DRIVE and USE_CACHE and valid_gguf(drive_model_path()):
        print("⏳ Restoring the finished model from Google Drive...")
        shutil.copyfile(drive_model_path(), FINAL_GGUF + ".part")
        os.replace(FINAL_GGUF + ".part", FINAL_GGUF)
        cleanup({os.path.basename(FINAL_GGUF)})
        return FINAL_GGUF, QUANT
    print("✅ Conversion deps: " + get_convert_deps())
    src, requant = None, False
    if valid_gguf(F16_GGUF):
        print("✅ F16 GGUF already converted — resuming at quantization.")
        src = F16_GGUF
    elif valid_gguf(Q8_GGUF):
        print("✅ Q8_0 GGUF (rescue path) already converted — resuming at quantization.")
        src, requant = Q8_GGUF, True
    else:
        print(f"⏳ Downloading the BF16 safetensors of {REPO_ID} (~29.5 GB)...")
        download_bf16()
        how = convert_stage()
        shutil.rmtree(BF16_DIR, ignore_errors=True)
        print("✅ Deleted the BF16 safetensors dir (~29.5 GB freed).")
        src, requant = (F16_GGUF, False) if how == "f16" else (Q8_GGUF, True)
    quantize_stage(src, requantize=requant)
    cleanup({os.path.basename(FINAL_GGUF)})
    print(f"✅ Pipeline done in {int((time.time()-t0)//60)} min {int((time.time()-t0)%60):02d} s | "
          f"{shutil.disk_usage(SCRATCH).free/1e9:.1f} GB free.")
    save_model_to_drive()
    return FINAL_GGUF, QUANT

def main():
    global ARCH, USE_CACHE, CACHED_BIN, CACHED_QUANT
    mount_drive()
    USE_CACHE = USE_DRIVE_CACHE
    ARCH = gpu_check()
    free_gb = shutil.disk_usage(SCRATCH).free / 1e9
    if free_gb < 35:
        raise RuntimeError(
            f"❌ Only {free_gb:.0f} GB free in {SCRATCH}. The pipeline needs ~35 GB at the very least (≈60 GB comfortably). "
            "Runtime → Disconnect and delete runtime → reconnect to get a fresh disk, then run again.")
    if free_gb < 62:
        print(f"⚠️ {free_gb:.0f} GB free in {SCRATCH}; the conversion peaks at ~60 GB. If the disk fills up, the cell "
              f"automatically switches to a smaller-footprint route (Q8_0 → {QUANT}).")
    if USE_CACHE:
        os.makedirs(CACHE_DIR, exist_ok=True)
    CACHED_BIN   = f"{CACHE_DIR}/llama-server-{LLAMA_TAG}-sm{ARCH}"
    CACHED_QUANT = f"{CACHE_DIR}/llama-quantize-{LLAMA_TAG}-sm{ARCH}"

    # checkout first (seconds): the compiler AND the converter both need it, and nothing may delete it later
    print("✅ llama.cpp source: " + ensure_llama_source() + f" ({LLAMA_TAG})")
    if not valid_gguf(FINAL_GGUF):
        check_gated_access()
        print("✅ Hugging Face: gated-repo access OK (HF_TOKEN works)")

    t0 = time.time()
    print(f"⏳ Building llama.cpp {LLAMA_TAG} and preparing {REPO_ID} ({QUANT} from BF16) at the same time...")
    with ThreadPoolExecutor(max_workers=4) as ex:
        jobs = {"build": ex.submit(build_job), "model": ex.submit(prep_model),
                "tunnel tool": ex.submit(get_cloudflared), "web tools": ex.submit(get_web_deps)}
        wait_jobs(jobs, t0)
    build_status = jobs["build"].result()
    model_path, quant_label = jobs["model"].result()
    cf_status = jobs["tunnel tool"].result()
    web_status = jobs["web tools"].result()
    try:
        ui_status = prepare_webui()
    except Exception as e:                      # noqa: BLE001
        ui_status = f"will be fetched by Cell 2 ({str(e)[:80]})"

    cfg = {"model": model_path, "quant": quant_label, "bin": BIN_PATH, "cloudflared": CF_PATH,
           "tag": LLAMA_TAG, "ui_tag": LLAMA_TAG, "webui_src": WEBUI_SRC}
    json.dump(cfg, open(CONFIG, "w"))
    print(f"\n✅ Server:      {build_status} (llama.cpp {LLAMA_TAG})")
    print(f"✅ Model:       {os.path.basename(model_path)} ({os.path.getsize(model_path)/1e9:.1f} GB, {quant_label})")
    print(f"✅ Tunnel tool: {cf_status}")
    print(f"✅ Web tools:   {web_status}")
    print(f"✅ Chat UI:     {ui_status}")
    print(f"⏱  {int(time.time() - t0)//60} min {int(time.time() - t0)%60} s.  👉 Now run Cell 2.")

if __name__ == "__main__":
    main()


In [ ]:
#@title 2️⃣ Launch — choose context, reasoning and web search, then run (keep this cell running!) { display-mode: "form" }
CONTEXT_WINDOW = "Max · largest that fits one T4 (default)" #@param ["Max · largest that fits one T4 (default)", "65536 · YaRN-extended", "49152 · YaRN-extended", "32768 · native, best quality", "24576", "16384", "8192"]
KV_CACHE = "auto · 8-bit chat memory when it fits, else 4-bit (default)" #@param ["auto · 8-bit chat memory when it fits, else 4-bit (default)", "q8_0 · 8-bit only: higher quality, less context", "q4_0 · 4-bit: the most context"]
REASONING = "Max · unlimited thinking (default)" #@param ["Max · unlimited thinking (default)", "High · up to 8192 thinking tokens", "Medium · up to 2048 thinking tokens", "Low · up to 512 thinking tokens", "Off · answers directly, no thinking"]
WEB_SEARCH = "Parallel · free, no key needed (default)" #@param ["Parallel · free, no key needed (default)", "Exa · free tier is rate-limited, best with a key", "DuckDuckGo · free, no key needed", "Off · no web tools"]
AUTO_FALLBACK = True #@param {type:"boolean"}
PARALLEL_API_KEY = "VtYAbqEv6_sOvxUtcGeHI28jHmIpGuxM_talrLr6" #@param {type:"string"}
EXA_API_KEY = "29334ebe-08ca-4c98-8b57-af84986d1464" #@param {type:"string"}
PUBLIC_LINK = "Cloudflare · no account, new link each run (default)" #@param ["Cloudflare · no account, new link each run (default)", "Off · private Colab link only"]
SHOW_INLINE = True #@param {type:"boolean"}
# CONTEXT_WINDOW: "Max" = the biggest window that physically fits this GPU (the chat memory/KV cache drops from
#   8-bit to 4-bit only when that is what it takes). A fixed number is honoured the same way: 8-bit memory if it fits.
# KV_CACHE: the chat memory (KV cache). 8-bit is almost lossless; 4-bit doubles the context but is a bit less precise.
# REASONING: these models have no built-in thinking mode, so the notebook gives them one: a reasoning protocol is added to
#   the system prompt (the model thinks inside <think>...</think>, shown in a collapsible panel) and, for
#   High/Medium/Low, llama-server also enforces a hard thinking budget. Max = unlimited thinking.
# AUTO_FALLBACK: if the chosen search service is down or rate-limited, the other free ones are tried automatically.
# API keys are hardcoded above (source of truth). Colab secrets named PARALLEL_API_KEY / EXA_API_KEY are only
#   used if the field above is emptied. A Parallel key raises limits and adds a deep "research" tool.
# PUBLIC_LINK: anyone with the link can use your session. Cloudflare needs no account; the address changes each run.
# SHOW_INLINE: also shows the chat right here in the notebook (private, always works).

# ---------------- advanced (rarely needed) ----------------
TEMP, TOP_P, TOP_K, MIN_P = 0.7, 0.9, 40, 0.05   # sampling: calm settings = reliable tool calls + long answers
REPEAT_PENALTY   = 1.05
PRESENCE_PENALTY = 0.0         # if answers ever loop/repeat, set 1.0
USE_BUNDLED_TEMPLATE = True    # tool-capable chat template shipped in this notebook (see Troubleshooting)
KEEP_TOKENS      = 1024        # when a chat outgrows the window, the first N tokens (system prompt + tools) are never dropped
MIN_FREE_MIB     = 350         # GPU memory that must stay free after loading, so long chats never crash later
MAX_SERVER_RESTARTS = 5
# -----------------------------------------------------------
import os, re, sys, json, time, datetime, subprocess, shutil, socket
import requests
from IPython.display import display, HTML

PROFILE = dict(
    NAME="Cyber-Quack-14B",
    N_LAYER=48, N_KV_HEAD=8, HEAD_DIM=128,           # Qwen2.5-14B (re-read from the GGUF at run time)
    YARN_BASE=32768,                                 # Qwen2.5 trained window; larger contexts use YaRN
    CTX_LADDER=[65536, 49152, 32768, 24576, 16384, 12288, 8192, 4096],
    OVERHEAD_MIB=900,                                # CUDA context + compute buffers of a 14B model on a T4
    CONFIG="/content/cyberquack_config.json",
    TEMPLATE_STYLE="qwen", DEFAULT_SYSTEM="You are Cyber-Quack, a helpful security research assistant.",
)

# ==== PURE-BEGIN  (no side effects: unit-tested outside Colab) ====
import os, re, json, struct

# ---------------------------------------------------------------- chat templates
# One template body, two tool-response styles. Both render the tools block + <tool_call> format that
# llama.cpp's Hermes-2-Pro parser understands, and both know about `enable_thinking`, so the server can
# split <think>...</think> into the reasoning panel (and --reasoning off closes the think block at once).
_TEMPLATE_HEAD = r"""{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages and messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- '__DEFAULT_SYSTEM__' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages and messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
    {%- else %}
        {{- '<|im_start|>system\n__DEFAULT_SYSTEM__<|im_end|>\n' }}
    {%- endif %}
{%- endif %}
{%- for message in messages %}
    {%- if (message.role == "user") or (message.role == "system" and not loop.first) or (message.role == "assistant" and not message.tool_calls) %}
        {{- '<|im_start|>' + message.role + '\n' + message.content + '<|im_end|>' + '\n' }}
    {%- elif message.role == "assistant" %}
        {{- '<|im_start|>' + message.role }}
        {%- if message.content %}
            {{- '\n' + message.content }}
        {%- endif %}
        {%- for tool_call in message.tool_calls %}
            {%- if tool_call.function is defined %}
                {%- set tool_call = tool_call.function %}
            {%- endif %}
            {{- '\n<tool_call>\n{"name": "' }}
            {{- tool_call.name }}
            {{- '", "arguments": ' }}
            {{- tool_call.arguments | tojson }}
            {{- '}\n</tool_call>' }}
        {%- endfor %}
        {{- '<|im_end|>\n' }}
"""
_TOOL_QWEN = r"""    {%- elif message.role == "tool" %}
        {%- if (loop.index0 == 0) or (messages[loop.index0 - 1].role != "tool") %}
            {{- '<|im_start|>user' }}
        {%- endif %}
        {{- '\n<tool_response>\n' }}
        {{- message.content }}
        {{- '\n</tool_response>' }}
        {%- if loop.last or (messages[loop.index0 + 1].role != "tool") %}
            {{- '<|im_end|>\n' }}
        {%- endif %}
    {%- endif %}
"""
_TOOL_HERMES = r"""    {%- elif message.role == "tool" %}
        {%- if (loop.index0 == 0) or (messages[loop.index0 - 1].role != "tool") %}
            {{- '<|im_start|>tool' }}
        {%- endif %}
        {{- '\n<tool_response>\n' }}
        {{- message.content }}
        {{- '\n</tool_response>' }}
        {%- if loop.last or (messages[loop.index0 + 1].role != "tool") %}
            {{- '<|im_end|>\n' }}
        {%- endif %}
    {%- endif %}
"""
_TEMPLATE_TAIL = r"""{%- endfor %}
{%- if add_generation_prompt %}
    {{- '<|im_start|>assistant\n' }}
    {%- if enable_thinking is defined and enable_thinking is false %}
        {{- '<think>\n\n</think>\n\n' }}
    {%- endif %}
{%- endif %}
"""

def build_chat_template(style, default_system):
    body = _TEMPLATE_HEAD + (_TOOL_HERMES if style == "hermes" else _TOOL_QWEN) + _TEMPLATE_TAIL
    return body.replace("__DEFAULT_SYSTEM__", default_system.replace("'", ""))


# ---------------------------------------------------------------- minimal GGUF header reader
_SCALARS = {0: "<B", 1: "<b", 2: "<H", 3: "<h", 4: "<I", 5: "<i", 6: "<f", 7: "<?", 10: "<Q", 11: "<q", 12: "<d"}

def read_gguf_meta(path, want_prefixes=("general.", "tokenizer.chat_template"), buf_mb=96):
    """Return {key: value} for the small metadata keys (scalars + strings); big arrays are skipped.
    Pure python, no dependency. Raises ValueError if the file is not a GGUF."""
    with open(path, "rb") as f:
        buf = f.read(buf_mb << 20)
    if buf[:4] != b"GGUF":
        raise ValueError("not a GGUF file")
    ver = struct.unpack_from("<I", buf, 4)[0]
    if ver < 2:
        raise ValueError(f"unsupported GGUF version {ver}")
    n_kv = struct.unpack_from("<Q", buf, 16)[0]
    pos, out = 24, {}

    def rstr(p):
        n = struct.unpack_from("<Q", buf, p)[0]
        return buf[p + 8:p + 8 + n].decode("utf-8", "replace"), p + 8 + n

    def skip(t, p):
        if t in _SCALARS:
            return p + struct.calcsize(_SCALARS[t])
        if t == 8:
            n = struct.unpack_from("<Q", buf, p)[0]
            return p + 8 + n
        if t == 9:
            et, cnt = struct.unpack_from("<IQ", buf, p)
            p += 12
            if et in _SCALARS:
                return p + cnt * struct.calcsize(_SCALARS[et])
            for _ in range(cnt):
                p = skip(et, p)
            return p
        raise ValueError(f"bad GGUF value type {t}")

    for _ in range(n_kv):
        key, pos = rstr(pos)
        t = struct.unpack_from("<I", buf, pos)[0]
        pos += 4
        keep = key.startswith(want_prefixes) or key.endswith((".block_count", ".context_length",
               ".head_count_kv", ".head_count", ".embedding_length", ".key_length"))
        if keep and t in _SCALARS:
            out[key] = struct.unpack_from(_SCALARS[t], buf, pos)[0]
            pos += struct.calcsize(_SCALARS[t])
        elif keep and t == 8:
            out[key], pos = rstr(pos)
        else:
            pos = skip(t, pos)
    return out

def model_dims(meta, fallback):
    """(n_layer, n_kv_head, head_dim, n_ctx_train) from GGUF metadata, falling back to the profile."""
    arch = meta.get("general.architecture", "")
    g = lambda k: meta.get(f"{arch}.{k}")
    try:
        n_layer = int(g("block_count") or fallback["N_LAYER"])
        kvh = g("attention.head_count_kv")
        kvh = int(kvh) if isinstance(kvh, (int, float)) and kvh else fallback["N_KV_HEAD"]
        hd = g("attention.key_length")
        if not hd:
            emb, heads = g("embedding_length"), g("attention.head_count")
            hd = (int(emb) // int(heads)) if emb and heads else fallback["HEAD_DIM"]
        return n_layer, kvh, int(hd), int(g("context_length") or 0)
    except Exception:
        return fallback["N_LAYER"], fallback["N_KV_HEAD"], fallback["HEAD_DIM"], 0


# ---------------------------------------------------------------- GPU memory planner
KV_BYTES_PER_ELEM = {"f16": 2.0, "q8_0": 34 / 32, "q4_0": 18 / 32}   # ggml block sizes

def kv_mib(ctx, kv, n_layer, n_kv_head, head_dim):
    return 2 * n_layer * n_kv_head * head_dim * KV_BYTES_PER_ELEM[kv] * ctx / 1048576

def plan_candidates(model_mib, total_mib, ladder, target, kv_pref, dims, overhead_mib, min_free_mib):
    """Ordered (ctx, kv) launch attempts. Largest context first; at equal context the best KV precision first.
    Entries predicted to fit come first; the rest (a safety net if the estimate is too pessimistic) follow."""
    kv_order = {"auto": ["q8_0", "q4_0"], "q8_0": ["q8_0"], "q4_0": ["q4_0"]}[kv_pref]
    sizes = [c for c in ladder if c <= target]
    fit, nofit = [], []
    for c in sizes:
        for kv in kv_order:
            need = model_mib + kv_mib(c, kv, *dims) + overhead_mib + min_free_mib
            (fit if need <= total_mib else nofit).append((c, kv))
    return fit + [x for x in nofit if x[1] == kv_order[-1]][:3]

def yarn_args(ctx, yarn_base):
    """llama-server flags for contexts beyond the model's trained window (Qwen2.5 + YaRN)."""
    if not yarn_base or ctx <= yarn_base:
        return []
    scale = round(ctx / yarn_base * 2) / 2          # 1.5, 2.0, ...
    return ["--rope-scaling", "yarn", "--rope-scale", str(scale), "--yarn-orig-ctx", str(yarn_base)]

def tool_cap_chars(ctx):
    """Max characters one web-tool result may add to the conversation (~1/8 of the window, 3 chars/token)."""
    return int(min(30000, max(5000, ctx * 0.375)))


# ---------------------------------------------------------------- web UI tarball (any archive layout)
def extract_ui(tgz, dest):
    """Unpack llama-<tag>-ui.tar.gz into `dest`, whatever the top-level layout is: the directory that
    holds index.html (shallowest one) becomes `dest`."""
    import tarfile, shutil
    tmp = dest + ".tmp"
    shutil.rmtree(tmp, ignore_errors=True)
    os.makedirs(tmp)
    with tarfile.open(tgz) as t:
        try:
            t.extractall(tmp, filter="data")
        except TypeError:                       # Python without extraction filters
            t.extractall(tmp)
    root, best = None, 10**6
    for r, _dirs, files in os.walk(tmp):
        if "index.html" in files:
            depth = os.path.relpath(r, tmp).count(os.sep)
            if depth < best:
                best, root = depth, r
    if root is None:
        shutil.rmtree(tmp, ignore_errors=True)
        raise RuntimeError("the UI archive has no index.html")
    shutil.rmtree(dest, ignore_errors=True)
    shutil.copytree(root, dest)
    shutil.rmtree(tmp, ignore_errors=True)
    return dest


# ---------------------------------------------------------------- choose a GGUF file in a repo listing
def pick_gguf(files, quant, preference, exact_name=None):
    """-> (filename, quant_used) or None. Exact name first, then the requested quant, then the next best."""
    ggufs = sorted(f for f in files if f.lower().endswith(".gguf") and "mmproj" not in f.lower()
                   and "/" not in f and not re.search(r"-\d{5}-of-\d{5}", f))
    if exact_name and exact_name in ggufs:
        return exact_name, quant
    order = preference[preference.index(quant):] if quant in preference else list(preference)
    for q in order:
        pat = re.compile(r"(?<![A-Za-z0-9])" + re.escape(q) + r"(?![A-Za-z0-9])", re.I)
        hit = [f for f in ggufs if pat.search(f)]
        if hit:
            return hit[0], q
    return None
# ==== PURE-END ====


LOG, TLOG = "/content/llama-server.log", "/content/tunnel.log"
CONFIG = PROFILE["CONFIG"]
WEB_SCRIPT, MCP_CONFIG = "/content/colab_web_mcp.py", "/content/colab_mcp.json"
TEMPLATE_FILE = "/content/chat_template_tools.jinja"
WEBUI_DIR = "/content/webui"
MiB = 1048576
LOCAL = requests.Session()
LOCAL.trust_env = False       # talk to the local server/tunnel directly, never through a proxy

def free_port(start):
    """First port from `start` that nothing else is using."""
    for p in range(start, start + 100):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sk:
            try:
                sk.bind(("127.0.0.1", p))
                return p
            except OSError:
                continue
    raise RuntimeError(f"No free port found near {start}")

def secret(name, typed):
    if typed.strip():
        return typed.strip()
    try:
        from google.colab import userdata
        return (userdata.get(name) or "").strip()
    except Exception:
        return ""

def gpu_mem():
    """(used MiB, total MiB) summed over GPUs, or (None, None)."""
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total",
                              "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout
        used = total = 0
        for line in out.strip().splitlines():
            u, t = line.split(",")
            used += int(u); total += int(t)
        return (used, total) if total else (None, None)
    except Exception:
        return None, None

TOGGLE_JS = r'''/* Injected by the notebook (tpuchat-v2): "Allow tool calls" switch inside the chat box,
   right after the + button. Applies instantly - never reloads the page, so a running
   answer (text or reasoning) is never interrupted. The choice is saved per browser. */
(function () {
  "use strict";
  if (window.__tpuchatAllowTools) return;
  window.__tpuchatAllowTools = true;

  var STATE_KEY = "tpuchat.allowTools";
  var TOOL_NAMES = ["web_search", "web_fetch", "web_research"];
  /* localStorage lists the official UI reads exactly once, at startup */
  var STORE_KEYS = ["LlamaUi.alwaysAllowedTools", "LlamaCppWebui.alwaysAllowedTools"];
  /* permission keys the UI derives for these tools (toolKey(): server:/mcp-<id>:/custom:) */
  var PERM_KEYS = [];
  var JUNK_KEYS = [];
  TOOL_NAMES.forEach(function (n) {
    PERM_KEYS.push("server:" + n, "mcp-web:" + n, "mcp-web:" + n.replace(/^web_/, ""), "custom:" + n);
  });
  ["web_web_search", "web_web_fetch", "web_web_research"].forEach(function (n) {
    JUNK_KEYS.push("server:" + n, "mcp-web:" + n, "custom:" + n);
  });
  TOOL_NAMES.forEach(function (n) {
    JUNK_KEYS.push("browser:" + n, "mcp:" + n);
  });

  function isOn() {
    try { var v = localStorage.getItem(STATE_KEY); return v === null ? true : v === "1"; }
    catch (e) { return true; }
  }
  function readList(k) {
    try {
      var a = JSON.parse(localStorage.getItem(k) || "[]");
      return Array.isArray(a) ? a.filter(function (x) { return typeof x === "string"; }) : [];
    } catch (e) { return []; }
  }
  function writeList(k, list) {
    try { localStorage.setItem(k, JSON.stringify(list)); } catch (e) {}
  }
  function cleanAndWrite(k, set) {
    JUNK_KEYS.forEach(function (p) { delete set[p]; });
    writeList(k, Object.keys(set));
  }
  function syncPerms(on) {
    STORE_KEYS.forEach(function (k) {
      var set = {};
      readList(k).forEach(function (x) { set[x] = 1; });
      PERM_KEYS.forEach(function (p) { if (on) set[p] = 1; else delete set[p]; });
      cleanAndWrite(k, set);
    });
  }
  function allowSaved(tool) {
    var keys = ["server:" + tool, "mcp-web:" + tool, "mcp-web:" + tool.replace(/^web_/, ""), "custom:" + tool];
    STORE_KEYS.forEach(function (k) {
      var set = {};
      readList(k).forEach(function (x) { set[x] = 1; });
      keys.forEach(function (p) { set[p] = 1; });
      cleanAndWrite(k, set);
    });
  }
  /* runs before the UI boots: the app picks this list up when it initializes */
  syncPerms(isOn());

  /* -------------- live plumbing (no reload: streaming is never interrupted) -------------- */

  var sessionAll = false;            /* "allow this session" covers all web tools until OFF again */
  var seenCard = { tool: "", t: 0 }; /* built-in permission card the user may act on */

  function isWebTool(t) { return typeof t === "string" && /^web_/.test(t); }

  /* Fetch gate - active only while the switch is OFF. Covers the one case localStorage
     cannot: web tools the UI still holds in its in-memory allowed set from an earlier
     boot, which would otherwise run silently instead of asking. */
  var origFetch = window.fetch.bind(window);
  window.fetch = function () {
    try {
      var input = arguments[0], init = arguments[1];
      var url = typeof input === "string" ? input : (input && input.url) || "";
      var method = ((init && init.method) || (input && input.method) || "GET").toUpperCase();
      if (method === "POST" && /\/tools\/?$/.test(String(url).split("?")[0]) &&
          init && typeof init.body === "string") {
        var tool = null;
        try { tool = (JSON.parse(init.body) || {}).tool; } catch (e) {}
        if (isWebTool(tool) && !isOn() && !sessionAll &&
            !(seenCard.tool === tool && Date.now() - seenCard.t < 8000)) {
          return gate(tool, arguments);
        }
      }
    } catch (e) { /* never break the app */ }
    return origFetch.apply(null, arguments);
  };

  var activeFinish = null;   /* decision callback of the currently open prompt */
  var gateChain = Promise.resolve();
  function gate(tool, args) {
    var run = function () {
      return new Promise(function (resolve) {
        var settled = false;
        var signal = args[1] && args[1].signal;
        function finish(allow, remember) {
          if (settled) return;
          settled = true;
          activeFinish = null;
          hideGateCard();
          if (remember) sessionAll = true;
          if (remember === "saved") allowSaved(tool);
          if (allow) resolve(origFetch.apply(null, args));
          else resolve(new Response(JSON.stringify({
                   error: "Blocked by the user: the Allow tool calls switch is OFF in the chat box. Turn it ON (green) to run web tools without asking."
                 }), { status: 403, statusText: "Blocked",
                      headers: { "Content-Type": "application/json" } }));
        }
        function abortNow() {
          if (settled) return;
          settled = true;
          activeFinish = null;
          hideGateCard();
          resolve(Promise.reject(new DOMException("The tool call was aborted.", "AbortError")));
        }
        if (signal) {
          if (signal.aborted) { abortNow(); return; }
          signal.addEventListener("abort", abortNow, { once: true });
        }
        showGateCard(tool, finish);
      });
    };
    gateChain = gateChain.then(run, run);
    return gateChain;
  }

  function esc(s) {
    return String(s).replace(/[&<>"']/g, function (c) {
      return { "&": "&amp;", "<": "&lt;", ">": "&gt;", '"': "&quot;", "'": "&#39;" }[c];
    });
  }

  var gateEl = null;
  function showGateCard(tool, finish) {
    hideGateCard();
    var d = document.createElement("div");
    d.id = "tpu-gate-card";
    d.setAttribute("role", "dialog");
    d.setAttribute("aria-label", "Tool call permission");
    d.innerHTML = '<div class="t">The model wants to run <b>' + esc(tool) + "</b></div>" +
      '<div class="s">The "Allow tool calls" switch is OFF, so this call needs your OK.</div>' +
      '<div class="row">' +
      '<button type="button" data-a="once">Allow once</button>' +
      '<button type="button" data-a="session">Allow this session</button>' +
      '<button type="button" data-a="saved">Always allow</button>' +
      '<button type="button" data-a="deny" class="deny">Deny</button>' +
      "</div>";
    d.addEventListener("click", function (e) {
      var b = e.target && e.target.closest ? e.target.closest("button[data-a]") : null;
      if (!b) return;
      var a = b.getAttribute("data-a");
      finish(a !== "deny", a === "session" ? "session" : (a === "saved" ? "saved" : null));
    });
    (document.body || document.documentElement).appendChild(d);
    gateEl = d;
    activeFinish = finish;
  }
  function hideGateCard() {
    if (gateEl) { try { gateEl.remove(); } catch (e) {} gateEl = null; }
  }
  document.addEventListener("keydown", function (e) {
    if (e.key === "Escape" && activeFinish) activeFinish(false, null);
  }, true);

  /* Watch the built-in permission card ("Allow use of <tool> ..."). While the switch is
     ON, approve it automatically (only needed when the in-memory set is missing keys,
     e.g. the switch was turned ON mid-session). While OFF, just remember it so a call
     the user just approved by hand is never asked twice. */
  function scanForCard() {
    var btns = document.getElementsByTagName("button");
    for (var i = 0; i < btns.length; i++) {
      var b = btns[i];
      if ((b.textContent || "").trim() !== "Allow once") continue;
      var root = b.parentElement, cardText = "";
      for (var k = 0; k < 7 && root; k++, root = root.parentElement) {
        cardText = root.textContent || "";
        if (cardText.indexOf("Allow use of ") >= 0) break;
      }
      if (cardText.indexOf("Allow use of ") < 0) continue;
      var m = cardText.match(/Allow use of\s+([A-Za-z0-9_.-]+)/);
      if (m) { seenCard.tool = m[1]; seenCard.t = Date.now(); }
      if (isOn() && !b.dataset.tpuDone) {
        b.dataset.tpuDone = "1";
        try { b.click(); } catch (e) {}
      }
    }
  }

  /* -------------- the switch, rendered inside the composer after the + button -------------- */

  var CSS_TEXT =
    "#tpu-allow-tools{display:inline-flex;align-items:center;gap:7px;height:32px;padding:0 12px;" +
    "border-radius:999px;border:1px solid rgba(148,163,184,.35);background:rgba(100,116,139,.14);" +
    "color:inherit;opacity:.85;font:600 12px/1 ui-sans-serif,system-ui,-apple-system,sans-serif;" +
    "cursor:pointer;user-select:none;white-space:nowrap;flex:0 0 auto;" +
    "transition:border-color .15s,background .15s,opacity .15s}" +
    "#tpu-allow-tools:hover{opacity:1;border-color:rgba(148,163,184,.6)}" +
    "#tpu-allow-tools .dot{width:8px;height:8px;border-radius:50%;background:#64748b;flex:0 0 auto;" +
    "transition:background .15s,box-shadow .15s}" +
    "#tpu-allow-tools.on{border-color:rgba(34,197,94,.55);background:rgba(34,197,94,.13);opacity:1}" +
    "#tpu-allow-tools.on .dot{background:#22c55e;box-shadow:0 0 7px rgba(34,197,94,.8)}" +
    "#tpu-allow-tools.on .lbl{color:#16a34a}" +
    "#tpu-allow-tools .lbl-short{display:none}" +
    "#tpu-gate-card{position:fixed;left:50%;bottom:96px;transform:translateX(-50%);z-index:2147483000;" +
    "width:min(440px,calc(100vw - 24px));padding:14px 16px;border-radius:14px;color:#e2e8f0;" +
    "background:#0f172a;border:1px solid rgba(148,163,184,.3);box-shadow:0 12px 40px rgba(0,0,0,.45);" +
    "font:400 13px/1.5 ui-sans-serif,system-ui,-apple-system,sans-serif;animation:tpu-pop .16s ease-out}" +
    "#tpu-gate-card .t{font-weight:600;margin-bottom:2px}" +
    "#tpu-gate-card .s{opacity:.75;margin-bottom:10px}" +
    "#tpu-gate-card .row{display:flex;flex-wrap:wrap;gap:8px}" +
    "#tpu-gate-card button{all:unset;cursor:pointer;padding:6px 12px;border-radius:8px;" +
    "font:600 12px/1 ui-sans-serif,system-ui,sans-serif;color:inherit;" +
    "border:1px solid rgba(148,163,184,.4);background:rgba(148,163,184,.12)}" +
    "#tpu-gate-card button:hover{background:rgba(148,163,184,.24)}" +
    "#tpu-gate-card button.deny{border-color:rgba(239,68,68,.5);color:#f87171}" +
    "@keyframes tpu-pop{from{opacity:0;transform:translateX(-50%) translateY(6px)}" +
    "to{opacity:1;transform:translateX(-50%) translateY(0)}}" +
    "@media (prefers-color-scheme: dark){#tpu-allow-tools.on .lbl{color:#4ade80}}" +
    "@media (prefers-color-scheme: light){#tpu-gate-card{background:#fff;color:#0f172a;" +
    "border-color:#cbd5e1;box-shadow:0 12px 40px rgba(15,23,42,.18)}}" +
    "@media (max-width:640px){#tpu-allow-tools{height:28px;padding:0 9px;gap:5px;font-size:11px}" +
    "#tpu-allow-tools .lbl-long{display:none}#tpu-allow-tools .lbl-short{display:inline}" +
    "#tpu-gate-card{bottom:84px}}";

  var chip = null;
  function paint() {
    if (!chip) return;
    var on = isOn();
    chip.classList.toggle("on", on);
    chip.setAttribute("aria-checked", on ? "true" : "false");
    var long = chip.querySelector(".lbl-long"), short = chip.querySelector(".lbl-short");
    if (long) long.textContent = "Allow tool calls: " + (on ? "ON" : "OFF");
    if (short) short.textContent = "Tools: " + (on ? "ON" : "OFF");
  }
  function buildChip() {
    try {
      var css = document.createElement("style");
      css.id = "tpu-allow-tools-css";
      css.textContent = CSS_TEXT;
      document.head.appendChild(css);
    } catch (e) {}
    var b = document.createElement("button");
    b.id = "tpu-allow-tools";
    b.type = "button";
    b.setAttribute("role", "switch");
    b.setAttribute("aria-label", "Allow tool calls");
    b.title = "ON: web tools (search / fetch) run without asking. OFF: each web tool call asks first. " +
      "Applies instantly - no reload, typing and generating continue undisturbed.";
    b.innerHTML = '<span class="dot"></span><span class="lbl lbl-long"></span><span class="lbl lbl-short"></span>';
    /* keep the composer textarea focused and never trigger the + menu or a form submit */
    ["mousedown", "pointerdown", "touchstart", "touchend"].forEach(function (ev) {
      b.addEventListener(ev, function (e) { e.preventDefault(); e.stopPropagation(); });
    });
    b.addEventListener("click", function (e) {
      e.preventDefault();
      e.stopPropagation();
      var next = !isOn();
      try { localStorage.setItem(STATE_KEY, next ? "1" : "0"); } catch (err) {}
      syncPerms(next);
      if (!next) sessionAll = false;   /* a fresh OFF period asks again */
      paint();
      /* turning ON while a call is waiting for permission: let it through */
      if (next && activeFinish) activeFinish(true, "session");
      try {
        window.dispatchEvent(new StorageEvent("storage", { key: STATE_KEY, newValue: next ? "1" : "0" }));
      } catch (err) {}
    });
    paint();
    return b;
  }
  function ensureChip() {
    var host = document.querySelector("button.file-upload-button");
    if (!host || !host.isConnected) return false;
    if (!chip) chip = buildChip();
    /* preferred spot: the left group that holds the + button, next to it */
    var group = host.closest(".mr-auto") || host.parentElement;
    if (!group) return false;
    if (chip.isConnected && chip.parentElement === group) return true;
    try { group.appendChild(chip); paint(); } catch (e) { return false; }
    return true;
  }

  window.addEventListener("storage", function (e) {
    if (e && e.key === STATE_KEY) { try { paint(); } catch (err) {} }
  });

  /* -------------- keep-alive: survive Svelte re-renders and route changes -------------- */

  var scanTimer = 0, chipTimer = 0;
  function scheduleScan() {
    if (scanTimer) return;
    scanTimer = setTimeout(function () { scanTimer = 0; try { scanForCard(); } catch (e) {} }, 120);
  }
  function scheduleChip() {
    if (chipTimer) return;
    chipTimer = setTimeout(function () { chipTimer = 0; try { ensureChip(); } catch (e) {} }, 150);
  }
  function start() {
    try {
      var mo = new MutationObserver(function (muts) {
        scheduleChip();
        for (var i = 0; i < muts.length; i++) {
          var nodes = muts[i].addedNodes;
          for (var j = 0; j < nodes.length; j++) {
            var n = nodes[j];
            if (n.nodeType === 1 && (n.textContent || "").indexOf("Allow use of ") >= 0) {
              scheduleScan();
              return;
            }
          }
        }
      });
      mo.observe(document.documentElement, { childList: true, subtree: true });
    } catch (e) {}
    setInterval(function () {
      try { ensureChip(); } catch (e) {}
      try { scanForCard(); } catch (e) {}
    }, 1500);
    try { ensureChip(); } catch (e) {}
    console.info("[tpuchat] Allow tool calls switch installed (in the chat box, next to +)");
  }
  if (document.readyState === "loading") document.addEventListener("DOMContentLoaded", start);
  else start();
})();'''


WEB_MCP_SOURCE = r'''#!/usr/bin/env python3
"""Colab web tools: a tiny stdio MCP server for llama-server (--mcp-servers-config).

Gives the model web_search / web_fetch (and web_research when a Parallel key is set), backed by:
  parallel   - Parallel MCP (https://mcp.parallel.ai/mcp), free without a key
  exa        - Exa MCP (https://mcp.exa.ai/mcp), free tier is rate limited, key optional
  duckduckgo - DuckDuckGo search (ddgs) + direct page download, no key
If the chosen backend fails (rate limit, network, timeout) the next one is tried, and EVERY call is
bounded by a hard deadline (WEB_DEADLINE, default 80 s): a slow backend can never leave the chat
hanging, and the answer always comes back before a proxy (Cloudflare: 100 s) would cut the request.

Settings come from environment variables set by the notebook:
  WEB_BACKEND        parallel | exa | duckduckgo        (default parallel)
  PARALLEL_API_KEY   optional, higher limits + adds web_research
  EXA_API_KEY        optional
  WEB_FALLBACK       1 = try other backends on failure  (default 1)
  WEB_MAX_CHARS      cap on characters returned per call (set by the notebook from the context size)
  WEB_DEADLINE       hard limit in seconds for one tool call (default 80)
Only JSON-RPC goes to stdout; diagnostics go to stderr.
"""
import json, os, re, sys, time, html, threading, urllib.request, urllib.error

BACKEND = os.environ.get("WEB_BACKEND", "parallel").strip().lower() or "parallel"
PARALLEL_KEY = os.environ.get("PARALLEL_API_KEY", "").strip()
EXA_KEY = os.environ.get("EXA_API_KEY", "").strip()
FALLBACK = os.environ.get("WEB_FALLBACK", "1") != "0"
MAX_CHARS = int(os.environ.get("WEB_MAX_CHARS", "12000") or 12000)
DEADLINE = float(os.environ.get("WEB_DEADLINE", "80") or 80)
HTTP_TIMEOUT = 25
UA = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130 Safari/537.36"


def log(*a):
    print("[colab-web]", *a, file=sys.stderr, flush=True)


# ----------------------------------------------------------------- remote MCP client
class RemoteMCP:
    """Minimal Streamable-HTTP MCP client (JSON or SSE replies)."""

    def __init__(self, url, headers=None, timeout=HTTP_TIMEOUT):
        self.url, self.timeout, self.sid = url, timeout, None
        self.headers = {"Content-Type": "application/json",
                        "Accept": "application/json, text/event-stream",
                        "User-Agent": "colab-llama/5"}
        self.headers.update(headers or {})
        self._id = 0

    def _post(self, payload, want_reply=True):
        h = dict(self.headers)
        if self.sid:
            h["Mcp-Session-Id"] = self.sid
        req = urllib.request.Request(self.url, data=json.dumps(payload).encode("utf-8"),
                                     headers=h, method="POST")
        try:
            with urllib.request.urlopen(req, timeout=self.timeout) as r:
                sid = r.headers.get("Mcp-Session-Id")
                if sid:
                    self.sid = sid
                ctype = (r.headers.get("Content-Type") or "").lower()
                body = r.read().decode("utf-8", errors="replace")   # always UTF-8 (no mojibake)
        except urllib.error.HTTPError as e:
            detail = ""
            try:
                detail = e.read().decode("utf-8", errors="replace")[:500]
            except Exception:
                pass
            raise RuntimeError(f"HTTP {e.code} from {self.url}: {detail}")
        if not want_reply:
            return None
        msgs = []
        if "event-stream" in ctype:
            data_lines = []
            for line in body.splitlines() + [""]:          # SSE: blank line ends an event
                if line.startswith("data:"):
                    data_lines.append(line[5:].lstrip(" "))
                elif line.strip() == "":
                    if data_lines:
                        try:
                            msgs.append(json.loads("\n".join(data_lines)))
                        except ValueError:
                            pass
                        data_lines = []
        elif body.strip():
            j = json.loads(body)
            msgs = j if isinstance(j, list) else [j]
        for m in msgs:
            if isinstance(m, dict) and m.get("id") == payload.get("id"):
                return m
        for m in msgs:                                      # some servers reply with id null on errors
            if isinstance(m, dict) and "error" in m:
                return m
        raise RuntimeError("No reply from MCP server")

    def _rpc(self, method, params=None):
        self._id += 1
        reply = self._post({"jsonrpc": "2.0", "id": self._id, "method": method, "params": params or {}})
        if "error" in reply:
            err = reply["error"]
            raise RuntimeError(err.get("message", str(err)) if isinstance(err, dict) else str(err))
        return reply.get("result", {})

    def call(self, tool, args):
        self._rpc("initialize", {"protocolVersion": "2025-06-18", "capabilities": {},
                                 "clientInfo": {"name": "colab-llama", "version": "5"}})
        try:
            self._post({"jsonrpc": "2.0", "method": "notifications/initialized"}, want_reply=False)
        except Exception:
            pass
        res = self._rpc("tools/call", {"name": tool, "arguments": args})
        text = result_text(res)
        if res.get("isError"):
            raise RuntimeError(text or "tool error")
        return text


def result_text(res):
    parts = []
    for c in res.get("content") or []:
        if isinstance(c, dict) and c.get("type") == "text" and c.get("text"):
            parts.append(c["text"])
    if not parts and res.get("structuredContent") is not None:
        parts.append(json.dumps(res["structuredContent"], ensure_ascii=False, indent=1))
    return "\n".join(parts).strip()


# ----------------------------------------------------------------- backends  (each takes: args, timeout)
def _queries(a):
    q = a.get("query") or ""
    qs = a.get("search_queries") or []
    if isinstance(qs, str):
        qs = [qs]
    qs = [str(x).strip() for x in qs if str(x).strip()]
    if q.strip() and q.strip() not in qs:
        qs.insert(0, q.strip())
    return qs[:5]


def _urls(a):
    u = a.get("urls") or a.get("url") or []
    if isinstance(u, str):
        u = re.split(r"[\s,]+", u)
    out = []
    for x in u:
        x = str(x).strip().strip("<>\"'")
        if x and not re.match(r"^https?://", x, re.I):
            x = "https://" + x
        if x and x not in out:
            out.append(x)
    return out[:10]


def _phdr():
    return {"Authorization": f"Bearer {PARALLEL_KEY}"} if PARALLEL_KEY else {}


def parallel_search(a, timeout):
    qs = _queries(a)
    objective = (a.get("objective") or "").strip() or (qs[0] if qs else "")
    return RemoteMCP("https://mcp.parallel.ai/mcp", _phdr(), timeout).call(
        "web_search", {"objective": objective[:4000], "search_queries": qs or [objective[:200]]})


def parallel_fetch(a, timeout):
    args = {"urls": _urls(a)}
    if a.get("objective"):
        args["objective"] = str(a["objective"])[:200]
    if a.get("full_content"):
        args["full_content"] = True
    return RemoteMCP("https://mcp.parallel.ai/mcp", _phdr(), timeout).call("web_fetch", args)


def parallel_research(a, timeout):
    args = {"input": str(a.get("question") or a.get("input") or a.get("query") or "")[:20000]}
    return RemoteMCP("https://mcp.parallel.ai/mcp", _phdr(), timeout).call("web_research", args)


def _exa(timeout):
    hdr = {"Authorization": f"Bearer {EXA_KEY}"} if EXA_KEY else {}
    return RemoteMCP("https://mcp.exa.ai/mcp", hdr, timeout)


def exa_search(a, timeout):
    qs = _queries(a)
    args = {"query": " ".join(qs[:1]) or a.get("objective", ""), "numResults": 8}
    if a.get("objective"):
        args["objective"] = str(a["objective"])[:4000]
    return _exa(timeout).call("web_search_exa", args)


def exa_fetch(a, timeout):
    return _exa(timeout).call("web_fetch_exa", {"urls": _urls(a),
                                                "maxCharacters": 60000 if a.get("full_content") else 8000})


def ddg_search(a, timeout):
    try:
        from ddgs import DDGS
    except ImportError:
        raise RuntimeError("ddgs is not installed (pip install ddgs)")
    out, seen = [], set()
    for q in _queries(a)[:3] or [a.get("objective", "")]:
        try:
            hits = DDGS(timeout=min(15, timeout)).text(q, max_results=8) or []
        except TypeError:                       # very old ddgs without the timeout kwarg
            hits = DDGS().text(q, max_results=8) or []
        for r in hits:
            url = r.get("href") or r.get("url")
            if not url or url in seen:
                continue
            seen.add(url)
            out.append(f"## {r.get('title','')}\nURL: {url}\n{r.get('body','')}")
    if not out:
        raise RuntimeError("DuckDuckGo returned no results")
    return "\n\n".join(out)


def _download(url, timeout):
    req = urllib.request.Request(url, headers={"User-Agent": UA, "Accept": "text/html,*/*;q=0.8"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        raw = r.read(3_000_000)
        cs = r.headers.get_content_charset() or "utf-8"
        ctype = (r.headers.get("Content-Type") or "").lower()
    try:
        txt = raw.decode(cs, errors="replace")
    except LookupError:
        txt = raw.decode("utf-8", errors="replace")
    if "html" not in ctype and "<html" not in txt[:2000].lower():
        return txt
    try:
        import trafilatura
        ext = trafilatura.extract(txt, url=url, include_links=False, include_tables=True,
                                  output_format="markdown")
        if ext and len(ext) > 200:
            return ext
    except Exception:
        pass
    txt = re.sub(r"(?is)<(script|style|noscript|svg|head)[^>]*>.*?</\1>", " ", txt)
    txt = re.sub(r"(?s)<[^>]+>", " ", txt)
    return re.sub(r"[ \t\r\f\v]+", " ", re.sub(r"\n\s*\n+", "\n\n", html.unescape(txt))).strip()


def direct_fetch(a, timeout):
    urls, parts = _urls(a), []
    if not urls:
        raise RuntimeError("no valid URL given")
    per = max(3000, MAX_CHARS // len(urls))
    for u in urls[:5]:
        try:
            parts.append(f"# {u}\n{_download(u, min(timeout, 20))[:per]}")
        except Exception as e:
            parts.append(f"# {u}\n(could not load: {e})")
    return "\n\n".join(parts)


SEARCH = {"parallel": parallel_search, "exa": exa_search, "duckduckgo": ddg_search}
FETCH = {"parallel": parallel_fetch, "exa": exa_fetch, "duckduckgo": direct_fetch}


def _call_bounded(fn, args, seconds):
    """Run fn(args, seconds) in a worker thread and give up after `seconds` (the worker is a daemon)."""
    box = {}

    def work():
        try:
            box["r"] = fn(args, seconds)
        except BaseException as e:          # noqa: BLE001 - reported to the caller
            box["e"] = e

    t = threading.Thread(target=work, daemon=True)
    t.start()
    t.join(max(0.5, seconds))
    if t.is_alive():
        raise TimeoutError(f"no answer within {seconds:.0f}s")
    if "e" in box:
        raise box["e"]
    return box["r"]


def run_with_fallback(table, args, budget=None):
    """Try the chosen backend, then the others; the WHOLE call is bounded by `budget` seconds."""
    t0 = time.time()
    budget = DEADLINE if budget is None else budget
    order = [BACKEND] + ([b for b in ("parallel", "duckduckgo", "exa") if b != BACKEND] if FALLBACK else [])
    errors = []
    todo = [b for b in order if b in table]
    for i, b in enumerate(todo):
        left = budget - (time.time() - t0)
        if left < 3:
            errors.append(f"{b}: skipped (out of time)")
            break
        # fair slice: a hung backend must never starve the fallbacks that come after it
        per = min(HTTP_TIMEOUT, max(3.0, (left - 1) / (len(todo) - i)))
        try:
            text = _call_bounded(table[b], args, per)
            if text and text.strip():
                note = "" if b == BACKEND else f"[results from {b}; {BACKEND} was unavailable]\n"
                return note + text
            raise RuntimeError("empty result")
        except Exception as e:              # noqa: BLE001
            msg = str(e) or type(e).__name__
            log(f"{b} failed: {msg[:300]}")
            errors.append(f"{b}: {msg[:160]}")
    raise RuntimeError("all web backends failed - " + " | ".join(errors))


def trim(text, limit):
    if len(text) <= limit:
        return text
    cut = text[:limit]
    nl = cut.rfind("\n")
    if nl > limit * 0.7:
        cut = cut[:nl]
    return cut + "\n\n[...truncated to fit the context window]"


# ----------------------------------------------------------------- MCP server side
TOOLS = [
    {"name": "search",
     "description": ("Search the web for current information. Returns titles, URLs and text excerpts. "
                     "Use for anything recent, factual, or that you are unsure about."),
     "inputSchema": {"type": "object", "properties": {
         "query": {"type": "string", "description": "Short keyword search query, 3-8 words."},
         "objective": {"type": "string", "description": "Optional: one sentence on what you want to find."}},
         "required": ["query"]}},
    {"name": "fetch",
     "description": "Read one or more web pages (e.g. URLs from web_search) and return their main text.",
     "inputSchema": {"type": "object", "properties": {
         "urls": {"type": "array", "items": {"type": "string"}, "description": "Page URLs to read (1-3)."},
         "objective": {"type": "string", "description": "Optional: what to look for on the pages."},
         "full_content": {"type": "boolean", "description": "Return the whole page instead of the relevant parts."}},
         "required": ["urls"]}},
]
if BACKEND == "parallel" and PARALLEL_KEY:
    TOOLS.append({"name": "research",
                  "description": ("Deep research on ONE broad question; returns a cited answer. Slow (up to ~80 s) "
                                  "and uses credits - prefer search + fetch unless the question needs many sources."),
                  "inputSchema": {"type": "object", "properties": {
                      "question": {"type": "string", "description": "The research question."}},
                      "required": ["question"]}})


def handle(msg):
    method, mid = msg.get("method"), msg.get("id")
    if mid is None:
        return None                                          # notification
    if method == "initialize":
        pv = (msg.get("params") or {}).get("protocolVersion") or "2025-06-18"
        return {"protocolVersion": pv, "capabilities": {"tools": {"listChanged": False}},
                "serverInfo": {"name": "colab-web", "version": "5.0"}}
    if method == "ping":
        return {}
    if method == "tools/list":
        return {"tools": TOOLS}
    if method == "tools/call":
        p = msg.get("params") or {}
        name, args = p.get("name"), p.get("arguments") or {}
        if isinstance(args, str):
            try:
                args = json.loads(args)
            except ValueError:
                args = {"query": args}
        if not isinstance(args, dict):
            args = {"query": str(args)}
        try:
            if name in ("search", "web_search"):
                if not _queries(args) and not args.get("objective"):
                    raise RuntimeError("web_search needs a 'query'")
                text = run_with_fallback(SEARCH, args)
            elif name in ("fetch", "web_fetch"):
                if not _urls(args):
                    raise RuntimeError("web_fetch needs at least one URL in 'urls'")
                text = run_with_fallback(FETCH, args)
            elif name in ("research", "web_research"):
                if not PARALLEL_KEY:
                    raise RuntimeError("web_research needs a Parallel API key")
                text = _call_bounded(parallel_research, args, max(10, DEADLINE - 3))
            else:
                raise RuntimeError(f"unknown tool {name}")
            return {"content": [{"type": "text", "text": trim(text, MAX_CHARS)}], "isError": False}
        except Exception as e:                  # noqa: BLE001
            note = ("Web tool error: " + (str(e) or type(e).__name__) +
                    "\n(Do not repeat the same call. Either try ONE differently-worded query, or answer now "
                    "from your own knowledge and say that the live lookup failed.)")
            return {"content": [{"type": "text", "text": note}], "isError": True}
    raise KeyError(method)


def main():
    for line in sys.stdin:
        line = line.strip()
        if not line:
            continue
        try:
            msg = json.loads(line)
        except ValueError:
            continue
        msgs = msg if isinstance(msg, list) else [msg]
        for m in msgs:
            if not isinstance(m, dict):
                continue
            try:
                res = handle(m)
                if res is None:
                    continue
                out = {"jsonrpc": "2.0", "id": m.get("id"), "result": res}
            except KeyError:
                out = {"jsonrpc": "2.0", "id": m.get("id"),
                       "error": {"code": -32601, "message": f"Method not found: {m.get('method')}"}}
            except Exception as e:              # noqa: BLE001
                out = {"jsonrpc": "2.0", "id": m.get("id"), "error": {"code": -32603, "message": str(e)}}
            sys.stdout.write(json.dumps(out, ensure_ascii=False) + "\n")
            sys.stdout.flush()


if __name__ == "__main__":
    main()
'''


def patch_webui(index_path):
    try:
        with open(index_path, encoding="utf-8") as f:
            html = f.read()
        tag = "<script>" + TOGGLE_JS + "</script>\n"
        if "</head>" in html:
            html = html.replace("</head>", tag + "</head>", 1)
        elif "<body" in html:
            html = html.replace("<body", tag + "<body", 1)
        else:
            html = tag + html
        with open(index_path, "w", encoding="utf-8") as f:
            f.write(html)
        return True
    except OSError:
        return False

def ensure_webui(cfg):
    """Copy the official chat UI (prepared by Cell 1, or downloaded now) and add the Allow-tool-calls switch."""
    src = cfg.get("webui_src") or "/content/webui_src"
    try:
        if not os.path.isfile(os.path.join(src, "index.html")):
            tag = cfg.get("ui_tag") or cfg.get("tag")
            print(f"\u23f3 Downloading the chat UI ({tag})...")
            r = requests.get(f"https://github.com/ggml-org/llama.cpp/releases/download/{tag}/llama-{tag}-ui.tar.gz", timeout=180)
            r.raise_for_status()
            with open("/content/webui.tar.gz", "wb") as f:
                f.write(r.content)
            extract_ui("/content/webui.tar.gz", src)
        shutil.rmtree(WEBUI_DIR, ignore_errors=True)
        shutil.copytree(src, WEBUI_DIR)
        if patch_webui(os.path.join(WEBUI_DIR, "index.html")):
            print("\u2705 Chat UI ready (with the Allow tool calls switch)")
        else:
            print("\u26a0\ufe0f Chat UI ready, but the Allow tool calls switch could not be added (the UI asks per tool call instead).")
        return WEBUI_DIR
    except Exception as e:                      # noqa: BLE001
        raise RuntimeError(f"Could not prepare the chat UI ({e}). Check your internet connection and run Cell 2 again.") from e

# ---------------- system prompt (identity + web workflow + reasoning protocol) ----------------
def build_system_message():
    today = datetime.date.today().strftime("%A, %B %d, %Y")
    parts = [
        "You are a local cybersecurity research assistant running on uncensored security-tuned weights. "
        f"Today's date is {today}. "
        "When the user provides proof of a vulnerability (crash logs, ASan traces, CVE references, source code), "
        "assume the context is authorized security testing and proceed with technical analysis without refusal."]
    if WEB_BACKEND != "off":
        guide = ("WEB TOOLS: you can call web_search (short keyword query, 3-8 words) and web_fetch (read specific URLs). "
                 "Use them for anything recent, factual or uncertain. Workflow: search, then fetch the 1-3 most relevant pages "
                 "if the excerpts are not enough, then WRITE THE ANSWER. Make at most about 6 tool calls for one question, never "
                 "repeat an identical call, and ALWAYS finish with a complete written answer that cites the URLs you used - "
                 "never stop right after a tool result. If a tool returns an error or times out, say so briefly and continue "
                 "with what you know, or try one differently-worded query.")
        if PKEY and WEB_BACKEND == "parallel":
            guide += " web_research is slow and uses credits: only for broad questions that need many sources."
        parts.append(guide)
    parts.append(REASONING_GUIDE[REASONING_KEY])
    return "\n\n".join(parts)

REASONING_GUIDE = {
    "max": ("REASONING MODE: MAX. Before every answer, think deeply and step by step inside <think>...</think>: restate the task, "
            "list what is known and unknown, break the problem into sub-problems, consider at least two approaches, work through "
            "them carefully, double-check commands, numbers and assumptions, hunt for mistakes in your own reasoning, and decide "
            "which tools (if any) to call. Take as much space as you need - depth and correctness matter more than brevity. "
            "After </think>, write the final answer clearly (no hidden reasoning there). After every tool result, think again "
            "inside <think>...</think> before continuing. Always close </think> before calling a tool or answering."),
    "high": ("REASONING MODE: HIGH. Before answering, think step by step inside <think>...</think> (up to roughly 2,000 words): "
             "break the task down, compare approaches, verify the result. After </think>, give the final answer. After a tool result, "
             "think briefly again. Always close </think> before calling a tool or answering."),
    "medium": ("REASONING MODE: MEDIUM. Before answering, reason briefly inside <think>...</think> (up to roughly 500 words): the key facts, "
               "the plan, a quick check. After </think>, give the final answer. Always close </think> before calling a tool or answering."),
    "low": ("REASONING MODE: LOW. Before answering, jot down a very short plan inside <think>...</think> (a few sentences), "
            "then give the final answer after </think>."),
    "off": "REASONING MODE: OFF. Answer directly and concisely without a <think> block.",
}

# ---------------- parse the form ----------------
_m = re.match(r"\d+", CONTEXT_WINDOW.strip())
TARGET_CONTEXT = int(_m.group()) if _m else PROFILE["CTX_LADDER"][0]      # "Max ..." -> the top of the ladder
KV_PREF = KV_CACHE.split()[0].lower()
if KV_PREF not in ("auto", "q8_0", "q4_0"):
    KV_PREF = "auto"
REASONING_KEY = REASONING.split()[0].lower()
if REASONING_KEY not in REASONING_GUIDE:
    REASONING_KEY = "max"
REASONING_BUDGET = {"max": -1, "high": 8192, "medium": 2048, "low": 512, "off": 0}[REASONING_KEY]
WEB_BACKEND = {"parallel": "parallel", "exa": "exa", "duckduckgo": "duckduckgo"}.get(WEB_SEARCH.split()[0].lower(), "off")
LINK_MODE = PUBLIC_LINK.split()[0].lower()
if LINK_MODE not in ("cloudflare", "off"):
    LINK_MODE = "cloudflare"

if not os.path.exists(CONFIG):
    raise RuntimeError("Run Cell 1 first.")
cfg = json.load(open(CONFIG))
BIN, MODEL_PATH, CF, QUANT = cfg["bin"], cfg["model"], cfg["cloudflared"], cfg["quant"]
for _p in (BIN, MODEL_PATH):
    if not os.path.exists(_p):
        raise RuntimeError(f"Missing {_p}. Run Cell 1 again.")
if LINK_MODE == "cloudflare" and not os.path.exists(CF):
    raise RuntimeError("cloudflared is missing. Run Cell 1 again (or set PUBLIC_LINK to Off).")

PKEY = secret("PARALLEL_API_KEY", PARALLEL_API_KEY) if WEB_BACKEND != "off" else ""
EKEY = secret("EXA_API_KEY", EXA_API_KEY) if WEB_BACKEND != "off" else ""

# ---------------- model facts (from the GGUF itself) ----------------
try:
    META = read_gguf_meta(MODEL_PATH)
except Exception as _e:                         # noqa: BLE001
    print(f"\u26a0\ufe0f Could not read the GGUF header ({_e}); using built-in model facts.")
    META = {}
N_LAYER, N_KV_HEAD, HEAD_DIM, CTX_TRAIN = model_dims(META, PROFILE)
DIMS = (N_LAYER, N_KV_HEAD, HEAD_DIM)
MODEL_MIB = os.path.getsize(MODEL_PATH) / MiB
ALIAS = f"{PROFILE['NAME']}-{QUANT}"

# ---------------- chat template ----------------
def prepare_template():
    emb = META.get("tokenizer.chat_template", "") or ""
    embedded_has_tools = "<tool_call>" in emb
    if USE_BUNDLED_TEMPLATE or not embedded_has_tools:
        with open(TEMPLATE_FILE, "w", encoding="utf-8") as f:
            f.write(build_chat_template(PROFILE["TEMPLATE_STYLE"], PROFILE["DEFAULT_SYSTEM"]))
        return True, ("bundled tool-calling template" +
                      ("" if embedded_has_tools else " (the model's own template has no tool support)"))
    return False, "the model's own template"

USE_TEMPLATE_FILE, TEMPLATE_NOTE = prepare_template()

# ---------------- web tools (stdio MCP server) ----------------
def write_mcp_config(ctx):
    open(WEB_SCRIPT, "w").write(WEB_MCP_SOURCE)
    env = {"WEB_BACKEND": WEB_BACKEND, "WEB_FALLBACK": "1" if AUTO_FALLBACK else "0",
           "PARALLEL_API_KEY": PKEY, "EXA_API_KEY": EKEY, "PYTHONIOENCODING": "utf-8",
           "WEB_MAX_CHARS": str(tool_cap_chars(ctx)), "WEB_DEADLINE": "80"}
    json.dump({"mcpServers": {"web": {"command": sys.executable, "args": ["-u", WEB_SCRIPT],
                                      "env": env, "timeout_ms": 120000}}}, open(MCP_CONFIG, "w"))
    return env

def probe_web(env):
    print(f"\u23f3 Testing web search ({WEB_BACKEND})... ", end="", flush=True)
    try:
        probe = "\n".join(json.dumps(m) for m in [
            {"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {"protocolVersion": "2025-06-18"}},
            {"jsonrpc": "2.0", "id": 2, "method": "tools/call",
             "params": {"name": "search", "arguments": {"query": "llama.cpp github"}}}]) + "\n"
        out = subprocess.run([sys.executable, "-u", WEB_SCRIPT], input=probe, capture_output=True, text=True,
                             timeout=120, env=dict(os.environ, **env)).stdout
        res = [json.loads(l) for l in out.splitlines() if l.strip().startswith("{")]
        r2 = next((r for r in res if r.get("id") == 2), {}).get("result", {})
        if r2 and not r2.get("isError"):
            txt = r2["content"][0]["text"]
            print("\u2705" + (f"  (note: {txt.splitlines()[0][1:-1]})" if txt.startswith("[results from") else ""))
        else:
            msg = (r2.get("content") or [{"text": "no reply"}])[0]["text"].splitlines()[0]
            print(f"\u26a0\ufe0f {msg[:300]}\n   The tools stay enabled and retry on every call.")
    except Exception as e:                      # noqa: BLE001
        print(f"\u26a0\ufe0f test failed ({e}); the tools stay enabled and retry on every call.")

# ---------------- model server ----------------
OPTIONAL_FLAGS = {"--fit", "--cache-ram", "--keep", "--sse-ping-interval", "--cors-origins", "--repeat-penalty",
                  "--min-p", "--reasoning", "--reasoning-budget", "--reasoning-budget-message", "--context-shift",
                  "--chat-template-file", "--ui-config", "--presence-penalty", "--rope-scaling", "--rope-scale",
                  "--yarn-orig-ctx", "--top-k", "--top-p", "--temp"}
DROPPED = set()          # optional flags this llama.cpp build rejected (learned at runtime)

def pkill(name):
    subprocess.run(["pkill", "-x", name], stderr=subprocess.DEVNULL)

def stop(proc):
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=20)
        except subprocess.TimeoutExpired:
            proc.kill(); proc.wait()

def server_command(ctx, kv):
    ui_config = json.dumps({"systemMessage": SYSTEM_MSG, "agenticMaxTurns": 100, "max_tokens": -1,
                            "showThoughtInProgress": True})
    cmd = [BIN, "-m", MODEL_PATH, "--host", "127.0.0.1", "--port", str(PORT), "--alias", ALIAS,
           "-ngl", "99", "-c", str(ctx), "--parallel", "1", "-fa", "on", "-ctk", kv, "-ctv", kv, "--jinja"]
    def opt(flag, *vals):
        if flag not in DROPPED:
            cmd.extend([flag] + [str(v) for v in vals])
    opt("--fit", "off")
    opt("--cache-ram", 2048)
    opt("--context-shift")
    opt("--keep", KEEP_TOKENS)
    opt("--sse-ping-interval", 10)
    opt("--cors-origins", "*")                  # MCP tools default CORS to localhost-only; the tunnel/iframe origins differ
    if USE_TEMPLATE_FILE and TEMPLATE_OK[0]:
        opt("--chat-template-file", TEMPLATE_FILE)
    opt("--ui-config", ui_config)
    opt("--temp", TEMP); opt("--top-p", TOP_P); opt("--top-k", TOP_K); opt("--min-p", MIN_P)
    opt("--repeat-penalty", REPEAT_PENALTY)
    if PRESENCE_PENALTY > 0:
        opt("--presence-penalty", PRESENCE_PENALTY)
    if REASONING_KEY == "off":
        opt("--reasoning", "off")
    else:
        opt("--reasoning", "on")
        if REASONING_BUDGET > 0:
            opt("--reasoning-budget", REASONING_BUDGET)
            opt("--reasoning-budget-message", " Okay, I have thought about this enough. Time to give the final answer.\n")
    y = yarn_args(ctx, PROFILE["YARN_BASE"])
    for i in range(0, len(y), 2):
        opt(y[i], y[i + 1])
    if WEB_BACKEND != "off":
        cmd += ["--mcp-servers-config", MCP_CONFIG]
    cmd += ["--path", WEBUI_DIR]
    return cmd

TEMPLATE_OK = [True]

def start_server(ctx, kv):
    if WEB_BACKEND != "off":
        write_mcp_config(ctx)
    return subprocess.Popen(server_command(ctx, kv), stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

def wait_ready(proc, timeout=600):
    t = time.time()
    while time.time() - t < timeout:
        if proc.poll() is not None:
            return False
        try:
            if LOCAL.get(f"http://127.0.0.1:{PORT}/health", timeout=3).status_code == 200:
                time.sleep(1)
                return proc.poll() is None
        except requests.RequestException:
            pass
        time.sleep(2)
    return False

def log_text():
    try:
        return open(LOG, errors="replace").read()
    except OSError:
        return ""

def unknown_flag():
    for line in log_text().splitlines():
        l = line.lower()
        if "invalid argument" in l or "error while handling argument" in l or "unknown argument" in l:
            m = re.search(r"(--?[a-z][a-z0-9-]*)", line)
            if m:
                return m.group(1)
    return ""

def fail_reason():
    log = log_text().lower()
    if any(s in log for s in ("out of memory", "cudamalloc failed", "failed to allocate", "unable to allocate",
                              "cuda error: out of memory")):
        return "out of memory"
    if "unknown model architecture" in log:
        return "model architecture not supported by this llama.cpp build"
    if "couldn't bind" in log or "address already in use" in log:
        return "port busy"
    if unknown_flag():
        return "bad argument: " + unknown_flag()
    for line in log.splitlines():
        if "template" in line and any(w in line for w in ("error", "failed", "exception", "invalid", "unable")):
            return "chat template error"
    if "flash" in log and ("not supported" in log or "requires" in log):
        return "flash-attention / KV-quantization problem"
    if "failed to load model" in log or "error loading model" in log:
        return "model failed to load"
    if "no space left on device" in log:
        return "out of memory"
    return "see log"

def fully_on_gpu():
    m = re.search(r"offloaded (\d+)/(\d+) layers to gpu", log_text().lower())
    return m is None or m.group(1) == m.group(2)

def mem_label(kv):
    return {"q8_0": "8-bit chat memory", "q4_0": "4-bit chat memory"}.get(kv, "16-bit chat memory")

def try_launch(ctx, kv):
    """Start the server for one (context, kv) choice, repairing recoverable start-up problems on the way."""
    global PORT
    for _ in range(8):
        proc = start_server(ctx, kv)
        if wait_ready(proc):
            return proc, ""
        reason = fail_reason()
        stop(proc)
        if reason.startswith("bad argument"):
            flag = reason.split(": ", 1)[1]
            if flag in OPTIONAL_FLAGS and flag not in DROPPED:
                DROPPED.add(flag)
                print(f"\n   (this llama.cpp build does not know {flag} - continuing without it) ", end="", flush=True)
                continue
        if reason == "port busy":
            PORT = free_port(PORT + 1)
            print(f"\n   (port was busy, switching to {PORT}) ", end="", flush=True)
            continue
        if reason == "chat template error" and USE_TEMPLATE_FILE and TEMPLATE_OK[0]:
            TEMPLATE_OK[0] = False
            print("\n   (the bundled chat template was rejected - using the model's own template) ", end="", flush=True)
            continue
        return None, reason
    return None, "too many start-up retries"

def launch_best(target):
    used0, total = gpu_mem()
    # Subtract at most 1 GiB of "already used" memory: right after a crash the old process may still show up in
    # nvidia-smi for a moment. If something really does hog the GPU, the load fails and the next step is tried.
    avail = (total - min(used0 or 0, 1024)) if total else 15000
    cands = plan_candidates(MODEL_MIB, avail, PROFILE["CTX_LADDER"], target, KV_PREF, DIMS,
                            PROFILE["OVERHEAD_MIB"], MIN_FREE_MIB)
    if not cands:
        cands = [(PROFILE["CTX_LADDER"][-1], "q4_0")]
    print(f"   plan: model {MODEL_MIB/1024:.1f} GiB + chat memory + {PROFILE['OVERHEAD_MIB']} MiB working space "
          f"on a {avail/1024:.1f} GiB GPU")
    for ctx, kv in cands[:8]:
        extra = f", YaRN \u00d7{ctx / PROFILE['YARN_BASE']:.1f}" if PROFILE["YARN_BASE"] and ctx > PROFILE["YARN_BASE"] else ""
        print(f"   trying {ctx:>7,} tokens ({mem_label(kv)}{extra}) ... ", end="", flush=True)
        proc, reason = try_launch(ctx, kv)
        if proc is not None and not fully_on_gpu():
            stop(proc); proc, reason = None, "model did not fully fit on the GPU"
        if proc is not None:
            used, tot = gpu_mem()
            if used is not None and tot - used < MIN_FREE_MIB:
                stop(proc); proc, reason = None, f"fits, but only {tot-used} MiB GPU memory left (unsafe for long chats)"
        if proc is not None:
            print("\u2705")
            return proc, ctx, kv
        print(f"\u274c ({reason})")
        if reason.startswith("bad argument") or reason in ("model failed to load", "chat template error",
                "model architecture not supported by this llama.cpp build"):
            break
        time.sleep(3)
    print(log_text()[-3000:])
    raise RuntimeError("Could not start the model - see the log above.")

# ---------------- self tests (non-fatal) ----------------
def smoke_test(ctx):
    base = f"http://127.0.0.1:{PORT}"
    try:
        p = LOCAL.get(base + "/props", timeout=10).json()
        n_ctx = (p.get("default_generation_settings") or {}).get("n_ctx")
        if n_ctx:
            print(f"   server reports a {n_ctx:,}-token window" + ("" if n_ctx >= ctx else f" (asked for {ctx:,})"))
    except Exception as e:                      # noqa: BLE001
        print(f"   (could not read /props: {str(e)[:80]})")
    if WEB_BACKEND != "off":
        try:
            names = [t.get("tool") or t.get("name") for t in LOCAL.get(base + "/tools", timeout=15).json()]
            web = [n for n in names if n and n.startswith("web_")]
            print(("   \u2705 web tools exposed to the model: " + ", ".join(sorted(web))) if web else
                  "   \u26a0\ufe0f the server lists no web_* tools - see /content/llama-server.log")
        except Exception as e:                  # noqa: BLE001
            print(f"   \u26a0\ufe0f could not list tools: {str(e)[:100]}")
    try:
        t0 = time.time()
        r = LOCAL.post(base + "/v1/chat/completions", timeout=180, json={
            "messages": [{"role": "user", "content": "Reply with the single word: ready"}],
            "max_tokens": 24, "temperature": 0, "chat_template_kwargs": {"enable_thinking": False}})
        r.raise_for_status()
        u = r.json().get("usage", {})
        print(f"   \u2705 generation works ({u.get('completion_tokens', '?')} tokens in {time.time()-t0:.1f} s)")
    except Exception as e:                      # noqa: BLE001
        print(f"   \u26a0\ufe0f test generation failed: {str(e)[:160]}")
        return
    if WEB_BACKEND != "off":
        try:
            tools = [{"type": "function", "function": {"name": "web_search", "description": "search the web",
                      "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}}]
            r = LOCAL.post(base + "/v1/chat/completions", timeout=180, json={
                "messages": [{"role": "user", "content": "Find the newest llama.cpp release."}],
                "tools": tools, "tool_choice": "required", "max_tokens": 160, "temperature": 0,
                "chat_template_kwargs": {"enable_thinking": False}})
            r.raise_for_status()
            calls = (r.json()["choices"][0]["message"].get("tool_calls") or [])
            print("   \u2705 tool-call parsing works" if calls else
                  "   \u26a0\ufe0f the model answered without a parsed tool call in the self-test (usually still fine in chat)")
        except Exception as e:                  # noqa: BLE001
            print(f"   \u26a0\ufe0f tool-call self-test failed: {str(e)[:200]}")

# ---------------- public link ----------------
def tunnel_ready():
    try:
        return LOCAL.get(f"http://127.0.0.1:{METRICS_PORT}/ready", timeout=3).status_code == 200
    except requests.RequestException:
        return False

def url_answers(url):
    try:
        return requests.get(url + "/health", timeout=8).status_code == 200
    except requests.RequestException:
        return False

def start_cloudflare():
    pkill("cloudflared")
    time.sleep(1)
    for attempt in range(3):
        p = subprocess.Popen([CF, "tunnel", "--url", f"http://127.0.0.1:{PORT}",
                              "--protocol", "http2", "--no-autoupdate",
                              "--metrics", f"127.0.0.1:{METRICS_PORT}"],
                             stdout=open(TLOG, "w"), stderr=subprocess.STDOUT)
        url = None
        for _ in range(60):
            if p.poll() is not None:
                break
            try:
                found = [u for u in re.findall(r"https://[-a-z0-9]+\.trycloudflare\.com", open(TLOG).read())
                         if not u.startswith("https://api.")]
            except OSError:
                found = []
            if found:
                url = found[0]
                break
            time.sleep(1)
        if url:
            for _ in range(30):   # wait until the link actually answers
                if p.poll() is not None:
                    break
                if tunnel_ready() and url_answers(url):
                    return p, url
                time.sleep(2)
            if p.poll() is None and tunnel_ready():
                return p, url          # connected; the edge may just need a few more seconds
        stop(p)
        time.sleep(3)
    return None, None

def link_alive(handle):
    return handle is not None and handle.poll() is None and tunnel_ready()

def show_link(url, title):
    display(HTML(f'<div style="font-size:16px;margin:8px 0">{title} <b><a href="{url}" target="_blank">{url}</a></b></div>'))

def server_healthy():
    try:
        return LOCAL.get(f"http://127.0.0.1:{PORT}/health", timeout=8).status_code == 200
    except requests.RequestException:
        return False

# ================= main =================
def main():
    global PORT, METRICS_PORT, SYSTEM_MSG
    pkill("llama-server")
    pkill("cloudflared")
    time.sleep(2)
    PORT, METRICS_PORT = free_port(8090), free_port(20241)
    SYSTEM_MSG = build_system_message()
    ensure_webui(cfg)
    web_env = write_mcp_config(TARGET_CONTEXT) if WEB_BACKEND != "off" else {}
    if WEB_BACKEND != "off":
        probe_web(web_env)

    print(f"\u23f3 Loading {PROFILE['NAME']} {QUANT} - target: " +
          ("the largest context that fits" if CONTEXT_WINDOW.strip().lower().startswith("max") else f"{TARGET_CONTEXT:,} tokens") + "...")
    print(f"   chat template: {TEMPLATE_NOTE}")
    server, CTX, KV = launch_best(TARGET_CONTEXT)

    used, total = gpu_mem()
    think_txt = ("OFF" if REASONING_KEY == "off" else "MAX (unlimited)" if REASONING_BUDGET < 0
                 else f"{REASONING.split()[0]} (up to {REASONING_BUDGET:,} tokens)")
    yarn = f", YaRN \u00d7{CTX / PROFILE['YARN_BASE']:.1f}" if PROFILE["YARN_BASE"] and CTX > PROFILE["YARN_BASE"] else ""
    print(f"\n\u2705 Model ready  |  model: {QUANT}  |  context: {CTX:,} tokens ({mem_label(KV)}{yarn})  |  reasoning: {think_txt}")
    if "Max" not in CONTEXT_WINDOW and CTX < TARGET_CONTEXT:
        print(f"   \u2139\ufe0f {TARGET_CONTEXT:,} didn't fit in GPU memory; using {CTX:,}.")
    if PROFILE["YARN_BASE"] and CTX > PROFILE["YARN_BASE"]:
        print(f"   \u2139\ufe0f Above {PROFILE['YARN_BASE']:,} tokens this model uses YaRN extension (quality at the far end of the window is slightly lower). "
              f"Pick 32768 in the form for the best quality.")
    web_txt = "OFF"
    if WEB_BACKEND != "off":
        web_txt = "ON \u00b7 " + WEB_BACKEND + (" + auto-fallback" if AUTO_FALLBACK else "")
    print(f"   web search: {web_txt}  |  one web result is capped at {tool_cap_chars(CTX):,} characters"
          f"  |  GPU memory: {used} / {total} MiB")
    smoke_test(CTX)
    print()

    tunnel, public_url = None, None
    try:
        from google.colab import output
        # Colab's private new-tab links (*.prod.colab.dev) return 404 in modern browsers,
        # so only the in-notebook view is shown; use the public link for a full tab.
        if SHOW_INLINE:
            print("\U0001F4AC Chat inside the notebook (private, only you):")
            output.serve_kernel_port_as_iframe(PORT, height=850)
        if LINK_MODE != "off":
            print(f"\u23f3 Opening the public link ({LINK_MODE})...")
            tunnel, public_url = start_cloudflare()
            if public_url:
                show_link(public_url, "\U0001F30D Public link:")
                print("   (If it says it can't be reached, wait 30 s and refresh, or open it in an incognito window.)")
            else:
                print("\u26a0\ufe0f The public link didn't start yet - it keeps retrying below. The chat above works meanwhile.")

        print("\nHow to use: open a link \u2192 type your message. Long text/code (1,000+ lines) is fine: paste it or attach the file.")
        if WEB_BACKEND != "off":
            print("Web search: use the \u201cAllow tool calls: ON / OFF\u201d switch inside the chat box, right after the + button, to control whether searches ask for permission (applies instantly, no reload).")
        print("Keep this cell running. Press \u25a0 (stop) to shut everything down.\n")

        start, last_report = time.time(), 0
        server_restarts = tunnel_restarts = bad_ticks = bad_health = 0
        while True:
            time.sleep(20)

            # 1) model server: restart on crash (smaller context if it ran out of memory) or if it stops answering
            crashed = server.poll() is not None
            if not crashed:
                bad_health = 0 if server_healthy() else bad_health + 1
            hung = bad_health >= 6                   # ~2 minutes without a single /health answer
            if crashed or hung:
                if server_restarts >= MAX_SERVER_RESTARTS:
                    print("\u274c Model server keeps failing. Last log lines:\n")
                    print(log_text()[-3000:])
                    break
                server_restarts += 1
                reason = "hung (no answer for ~2 min)" if hung else fail_reason()
                stop(server)
                target = CTX
                if reason == "out of memory":
                    smaller = [c for c in PROFILE["CTX_LADDER"] if c < CTX]
                    if smaller:
                        target = smaller[0]
                        print(f"\u26a0\ufe0f Ran out of GPU memory at {CTX:,} tokens - restarting with at most {target:,}.")
                print(f"\u26a0\ufe0f Model server stopped ({reason}) - restarting ({server_restarts}/{MAX_SERVER_RESTARTS})...")
                try:
                    server, CTX, KV = launch_best(target)
                    bad_health = 0
                    print("\u2705 Model server is back. Your chats are kept in your browser; just resend the last message.")
                except RuntimeError as e:
                    print(f"\u274c {e}")
                continue

            # 2) public link
            if LINK_MODE != "off":
                bad_ticks = 0 if link_alive(tunnel) else bad_ticks + 1
                if tunnel is None or bad_ticks >= 3:
                    tunnel_restarts += 1
                    print(f"\u26a0\ufe0f Public link dropped - reconnecting (#{tunnel_restarts})...")
                    stop(tunnel)
                    tunnel, new_url = start_cloudflare()
                    bad_ticks = 0
                    if new_url and new_url != public_url:
                        public_url = new_url
                        show_link(public_url, "\U0001F30D NEW public link (the old one no longer works):")
                    elif new_url:
                        print("\u2705 Public link reconnected (same address).")

            # 3) heartbeat every 10 min
            mins = int((time.time() - start) // 60)
            if mins - last_report >= 10:
                last_report = mins
                print(f"\u23f1 Up {mins} min | server restarts: {server_restarts} | link restarts: {tunnel_restarts}")
    except KeyboardInterrupt:
        print("Stopping...")
    finally:
        stop(tunnel)
        stop(server)
        print("Shut down.")

if __name__ == "__main__":
    main()


## Troubleshooting

- **"Access restricted / 401 / gated"** → `jabbatheduck/Cyber-Quack-14B` is a gated repo. Cell 1 checks the token *before* the build and prints these steps: accept the terms on the model page, create a READ token, add it as the secret `HF_TOKEN` (🔑 icon, *Notebook access* ON), then run Cell 1 again.
- **First run is slow (≈35–55 min)** → that's the BF16 download (~29.5 GB) + F16 conversion (~10–25 min) + Q5_K_M quantization (~8–15 min), and the llama.cpp compile (~15–25 min) running next to them. Re-runs skip every finished stage (the final GGUF, the F16 file and the BF16 folder are each detected and reused; files are written as `.part` and renamed when complete, so an interrupted run can never leave a half-written model behind).
- **Conversion ran out of disk** → the notebook auto-retries converting **straight to Q8_0** (≈14 GB less scratch) and then re-quantizes that to Q5_K_M. If even that fails, delete old files or use *Runtime → Disconnect and delete runtime* for a fresh disk.
- **Context is lower than you picked** → that is the most your GPU can hold with this model; Cell 2 tells you what it used. Pick a smaller context or the Q4_K_M quant (≈1.5 GiB more room).
- **"YaRN" note** → you are above Qwen2.5's native 32,768 window. Quality at the far end degrades slightly; pick **32768** for the best quality.
- **Chat stops after a web search / long answer** → v2 fixes the usual causes: web results are capped to a fraction of the context window, every web call is bounded to 80 s (so the answer always comes back before a proxy times out), the first 1,024 tokens (system prompt + tools) are never dropped when a long chat outgrows the window, and the chat UI may now take up to 100 agent steps. If it still happens, open `/content/llama-server.log` (the last lines say why) and use the **Allow tool calls** switch to see whether a tool is being blocked.
- **Web search says rate-limited / failed** → it falls back automatically (Parallel → DuckDuckGo → Exa). For heavy use, add a free key from platform.parallel.ai as a secret named `PARALLEL_API_KEY`.
- **Tool prompts keep appearing / never appear** → use the **Allow tool calls: ON / OFF** switch inside the chat box, right after the + button. On (green) = web tools run without asking; Off = every web tool call asks first. The choice is saved in your browser. If the switch is missing, hard-refresh once (Ctrl+Shift+R / Cmd+Shift+R).
- **The model rambles or loops** → set `PRESENCE_PENALTY = 1.0` in the “advanced” block of Cell 2, or lower the context.
- **No reasoning panel content** → the model thinks inside `<think>` tags (see the Reasoning note above). If a particular llama.cpp build does not split them out, the reasoning simply appears at the top of the answer. Choose **Off** to switch it off.
- **“the bundled chat template was rejected”** → Cell 2 prints this and continues with the model's own template; tool calls may then be less reliable. Set `USE_BUNDLED_TEMPLATE = False` to always use the model's own template.
- **“this llama.cpp build does not know --xyz”** → harmless: Cell 2 drops optional flags a build does not understand and carries on.
- **Public link doesn't open** → wait ~30 s and refresh. Cloudflare quick tunnels take a moment to propagate. The link changes every run; the monitoring loop reconnects it and prints the new address if it drops.
- **“q8_0 / 4-bit chat memory”** → that is the conversation cache (KV cache), not the model. The model quant is shown as `model:`.
- **Colab disconnects** → free Colab stops idle or long sessions (usually after a few hours). Re-run Cell 1 (fast: everything finished is reused, and the Drive cache skips the compile) and then Cell 2.
- **Download or build failed** → just run Cell 1 again; finished parts are reused.
- **Logs** → `/content/build.log`, `/content/llama-server.log`, `/content/tunnel.log` (and for Cyber-Quack `/content/convert.log`, `/content/quantize.log`).